In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir() and (path / 'outputs').is_dir())
df = pd.read_csv(REPO_ROOT / 'outputs' / 'iM뱅크데이터_거시경제지표포함.csv')

# M12 지속거래약화 최종 파이프라인

이 노트북은 `final0`의 8개 문서에서 잠근 최종 구조를 한 번에 실행하도록 구성한다.

실행 순서:

1. 원천 패널 감사
2. `Y_INTERVENE_M12_v2` 롤링 라벨 생성
3. R1 관계피처 16개와 D/A/C/K 동적피처 40개 생성
4. 2023 고정 참조분포와 KMeans 중심을 이용한 관계 세그먼트 배정
5. Purged Expanding-Window 검증
6. LightGBM + Isotonic 최종 Test
7. 2025.12 운영 위험점수와 설명정보 생성

최종 모델의 X에는 세그먼트 번호·업종이 들어가지 않는다. 이 정보는 설명과 집단별 안정성 감사에만 사용한다.


In [2]:
from pathlib import Path
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir() and (path / 'outputs').is_dir())
BASE_DIR = REPO_ROOT / 'outputs'
SOURCE_PATH = BASE_DIR / 'iM뱅크데이터_거시경제지표포함.csv'
OUTPUT_DIR = BASE_DIR / '수익성F_outputs'
MODEL_DIR = BASE_DIR / '수익성F_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE_SEGMENT = 20260714
RANDOM_STATE_MODEL = 20260718
ONSET_THRESHOLD = 0.60
PERSIST_THRESHOLD = 0.70
MODEL_CUTOFFS = pd.period_range('2023-12', '2025-06', freq='M')
OPERATING_CUTOFF = pd.Period('2025-12', freq='M')
STRICT_TARGET_LOCK = True
RUN_BOOTSTRAP = True
BOOTSTRAP_REPS = 1000

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

print('source:', SOURCE_PATH)
print('source sha256:', sha256_file(SOURCE_PATH))
print('outputs:', OUTPUT_DIR)


source: /Users/changeun_1/Desktop/iM/5. final/iM뱅크데이터_거시경제지표포함.csv
source sha256: 5c423106d629a5cf51c152b722a60ea450447453be58e832686871cde829965e
outputs: /Users/changeun_1/Desktop/iM/5. final/수익성F_outputs


## 1. 원천 패널 감사

문서 잠금값은 121,392행, 94열, 3,372개 법인, 2023.01~2025.12의 36개월 완전관측 패널이다. 금액의 0은 결측이 아니라 실제 비활성 상태로 유지한다.


In [3]:
PANEL = df.copy()
PANEL['기준년월'] = pd.to_numeric(PANEL['기준년월'], errors='raise').astype(int)
PANEL['월'] = pd.PeriodIndex(PANEL['기준년월'].astype(str), freq='M')
PANEL = PANEL.sort_values(['법인ID', '월']).reset_index(drop=True)

assert PANEL.columns.drop('월').size == 94, f'원천 열 수 불일치: {PANEL.columns.drop("월").size}'
assert not PANEL.duplicated(['법인ID', '월']).any(), '법인ID-월 중복이 있습니다.'

panel_audit = pd.Series({
    '행': len(PANEL),
    '원천_열': PANEL.shape[1] - 1,
    '법인': PANEL['법인ID'].nunique(),
    '월': PANEL['월'].nunique(),
    '시작월': str(PANEL['월'].min()),
    '종료월': str(PANEL['월'].max()),
    '법인별_최소월수': int(PANEL.groupby('법인ID')['월'].nunique().min()),
    '법인별_최대월수': int(PANEL.groupby('법인ID')['월'].nunique().max()),
})
display(panel_audit.to_frame('값'))

expected_panel = {'행': 121392, '원천_열': 94, '법인': 3372, '월': 36}
for key, value in expected_panel.items():
    assert panel_audit[key] == value, f'{key} 잠금값 불일치: {panel_audit[key]} != {value}'


,값
행,121392
원천_열,94
법인,3372
월,36
시작월,2023-01
종료월,2025-12
법인별_최소월수,36
법인별_최대월수,36


## 2. 거래축과 공통 생성 함수

- D: 요구불 입출금 — 필수축
- A: 자동이체 — 확인축
- C: 채널 — 확인축
- K: 카드 — 확인축

네 축은 원금액을 서로 합산해 Y를 만들지 않는다. 각 축을 자기 과거와 비교해 약화 여부를 만든 뒤 논리식으로 결합한다.


In [4]:
AXIS_COLS = {
    'D': ['요구불입금금액', '요구불출금금액'],
    'A': ['자동이체금액'],
    'C': ['창구거래금액', '인터넷뱅킹거래금액', '스마트뱅킹거래금액', '폰뱅킹거래금액', 'ATM거래금액'],
    'K': ['신용카드사용금액', '체크카드사용금액'],
}

DEPOSIT_BALANCE_COLS = ['요구불예금잔액', '거치식예금잔액', '적립식예금잔액', '수익증권잔액', '신탁잔액', '퇴직연금잔액']
OTHER_DEPOSIT_COLS = ['거치식예금잔액', '적립식예금잔액', '수익증권잔액', '신탁잔액', '퇴직연금잔액']
LOAN_PARENT_BALANCE_COLS = ['여신_운전자금대출잔액', '여신_시설자금대출잔액']
LOAN_DETAIL_BALANCE_COLS = [
    '운전_할인어음잔액', '운전_당좌대출잔액', '운전_일반자금대출잔액', '운전_무역금융잔액',
    '운전_기업구매자금대출잔액', '운전_외상매출채권담보대출잔액', '운전_기타운전자금대출잔액',
    '시설_일반자금대출잔액', '시설_에너지절약시설대출잔액', '시설_기타시설자금대출잔액',
]
CHANNEL_COLS = AXIS_COLS['C']
NONFACE_CHANNEL_COLS = ['인터넷뱅킹거래금액', '스마트뱅킹거래금액', '폰뱅킹거래금액', 'ATM거래금액']

DEPOSIT_COUNT_COLS = ['요구불예금좌수', '거치식예금좌수', '적립식예금좌수', '수익증권좌수', '신탁좌수', '퇴직연금좌수']
LOAN_PARENT_COUNT_COLS = ['여신_운전자금대출좌수', '여신_시설자금대출좌수']

ORDINAL_LABELS = [
    '0개', '1개', '2개', '2개초과 5개이하', '5개초과 10개이하',
    '10개초과 20개이하', '20개초과 30개이하', '30개초과 40개이하',
    '40개초과 50개이하', '50개 초과',
]
ORDINAL_MAP = {label: i / 9 for i, label in enumerate(ORDINAL_LABELS)}

def ordinal_rank(series):
    normalized = series.astype('string').str.replace('건', '개', regex=False)
    out = normalized.map(ORDINAL_MAP)
    if out.isna().any():
        unknown = sorted(normalized[out.isna()].dropna().unique().tolist())
        raise ValueError(f'정의되지 않은 순서형 범주: {unknown}')
    return out.astype(float)

def logdiff(a, b):
    return np.log1p(np.maximum(a, 0)) - np.log1p(np.maximum(b, 0))

def theil_sen_slopes(values):
    """12개월 행렬의 pairwise slope 중앙값. scipy 없이 결정적으로 계산한다."""
    values = np.asarray(values, dtype=float)
    log_values = np.log1p(np.maximum(values, 0))
    pairs = [(i, j) for i in range(log_values.shape[1] - 1) for j in range(i + 1, log_values.shape[1])]
    slopes = np.stack([(log_values[:, j] - log_values[:, i]) / (j - i) for i, j in pairs], axis=1)
    return np.median(slopes, axis=1)

def max_zero_run(values):
    inactive = np.asarray(values) <= 0
    run = np.zeros(inactive.shape[0], dtype=int)
    best = np.zeros(inactive.shape[0], dtype=int)
    for j in range(inactive.shape[1]):
        run = np.where(inactive[:, j], run + 1, 0)
        best = np.maximum(best, run)
    return best

def empirical_percentile(values, sorted_reference):
    values = np.asarray(values, dtype=float)
    ref = np.asarray(sorted_reference, dtype=float)
    left = np.searchsorted(ref, values, side='left')
    right = np.searchsorted(ref, values, side='right')
    return (left + right + 1.0) / (2.0 * len(ref))


In [5]:
# 고객 금액형을 숫자로 잠그고 음수·결측을 감사한다.
CUSTOMER_AMOUNT_COLS = sorted(set(sum(AXIS_COLS.values(), []) + DEPOSIT_BALANCE_COLS + LOAN_PARENT_BALANCE_COLS + LOAN_DETAIL_BALANCE_COLS))
for c in CUSTOMER_AMOUNT_COLS:
    PANEL[c] = pd.to_numeric(PANEL[c], errors='coerce')

amount_audit = pd.DataFrame({
    'missing': PANEL[CUSTOMER_AMOUNT_COLS].isna().sum(),
    'negative': (PANEL[CUSTOMER_AMOUNT_COLS] < 0).sum(),
})
display(amount_audit.query('missing > 0 or negative > 0'))

PANEL[CUSTOMER_AMOUNT_COLS] = PANEL[CUSTOMER_AMOUNT_COLS].fillna(0).clip(lower=0)

MONTHLY = PANEL[['법인ID', '월', '기준년월', '업종_대분류', '업종_중분류', '사업장_시도', '사업장_시군구', '법인_고객등급', '전담고객여부']].copy()
for axis, cols in AXIS_COLS.items():
    MONTHLY[f'M_{axis}'] = PANEL[cols].sum(axis=1)

MONTHLY['핵심거래활동합계'] = MONTHLY['M_D'] + MONTHLY['M_C'] + MONTHLY['M_K']
MONTHLY['수신자산합계'] = PANEL[DEPOSIT_BALANCE_COLS].sum(axis=1)
MONTHLY['기타수신합계'] = PANEL[OTHER_DEPOSIT_COLS].sum(axis=1)
MONTHLY['여신부모잔액합계'] = PANEL[LOAN_PARENT_BALANCE_COLS].sum(axis=1)
MONTHLY['수신상품폭'] = (PANEL[DEPOSIT_BALANCE_COLS] > 0).sum(axis=1)
MONTHLY['여신세부상품폭'] = (PANEL[LOAN_DETAIL_BALANCE_COLS] > 0).sum(axis=1)
MONTHLY['채널폭'] = (PANEL[CHANNEL_COLS] > 0).sum(axis=1)

active_domains = np.column_stack([
    MONTHLY['수신자산합계'].to_numpy() > 0,
    MONTHLY['여신부모잔액합계'].to_numpy() > 0,
    MONTHLY['M_D'].to_numpy() > 0,
    MONTHLY['M_C'].to_numpy() > 0,
    MONTHLY['M_K'].to_numpy() > 0,
    MONTHLY['M_A'].to_numpy() > 0,
])
MONTHLY['관계영역폭'] = active_domains.sum(axis=1)

MONTHLY['요구불대기타수신_로그균형_월'] = np.log1p(PANEL['요구불예금잔액']) - np.log1p(MONTHLY['기타수신합계'])
MONTHLY['운전대시설_로그균형_월'] = np.log1p(PANEL['여신_운전자금대출잔액']) - np.log1p(PANEL['여신_시설자금대출잔액'])

dc_total = MONTHLY['M_D'] + MONTHLY['M_C']
MONTHLY['입출금대채널_비중_월'] = np.divide(
    MONTHLY['M_D'], dc_total, out=np.full(len(MONTHLY), 0.5, dtype=float), where=dc_total.to_numpy() > 0
)
nonface = PANEL[NONFACE_CHANNEL_COLS].sum(axis=1)
MONTHLY['비대면대창구_로그균형_월'] = np.log1p(nonface) - np.log1p(PANEL['창구거래금액'])

deposit_rank_cols = []
for c in DEPOSIT_COUNT_COLS:
    rc = f'__rank_{c}'
    MONTHLY[rc] = ordinal_rank(PANEL[c])
    deposit_rank_cols.append(rc)
MONTHLY['수신좌수_rank_월'] = MONTHLY[deposit_rank_cols].mean(axis=1)

loan_rank_cols = []
for c in LOAN_PARENT_COUNT_COLS:
    rc = f'__rank_{c}'
    MONTHLY[rc] = ordinal_rank(PANEL[c])
    loan_rank_cols.append(rc)
MONTHLY['여신부모좌수_rank_월'] = MONTHLY[loan_rank_cols].mean(axis=1)
MONTHLY['신용카드개수_rank_월'] = ordinal_rank(PANEL['신용카드개수'])
MONTHLY['자동이체건수_rank_월'] = ordinal_rank(PANEL['자동이체거래건수'])
MONTHLY['카드활성_월'] = (MONTHLY['M_K'] > 0).astype(float)

FIRM_IDS = np.sort(MONTHLY['법인ID'].unique())
AXIS_PIVOTS = {
    axis: MONTHLY.pivot(index='법인ID', columns='월', values=f'M_{axis}').reindex(FIRM_IDS)
    for axis in AXIS_COLS
}
assert all(p.notna().all().all() for p in AXIS_PIVOTS.values())
print('월별 파생행렬:', MONTHLY.shape, '| 법인:', len(FIRM_IDS))


,missing,negative


월별 파생행렬: (121392, 38) | 법인: 3372


## 3. `Y_INTERVENE_M12_v2` 생성

축별 약화는 같은 축 안에서 다음 두 조건을 모두 만족해야 한다.

`ONSET_g`: `c+1~c+3` 각 월의 전년동월비가 모두 0.60 미만  
`PERSIST_g`: `mean(c+4:c+6) / mean(c-11:c) < 0.70`

최종 Y는 `W_D AND (W_A OR W_C OR W_K)`이다. 분모가 0인 축은 음성이 아니라 평가불가로 둔다.


In [6]:
def axis_weakening(pivot, cutoff, onset_threshold=ONSET_THRESHOLD, persist_threshold=PERSIST_THRESHOLD):
    pre_months = pd.period_range(cutoff - 11, cutoff, freq='M')
    onset_months = pd.period_range(cutoff + 1, cutoff + 3, freq='M')
    yoy_months = pd.period_range(cutoff - 11, cutoff - 9, freq='M')
    persist_months = pd.period_range(cutoff + 4, cutoff + 6, freq='M')

    base = pivot.loc[:, pre_months].mean(axis=1)
    denom = pivot.loc[:, yoy_months]
    future = pivot.loc[:, onset_months]
    denom_values = denom.to_numpy(float)
    ratios = np.divide(
        future.to_numpy(float), denom_values,
        out=np.full_like(denom_values, np.nan, dtype=float),
        where=denom_values > 0,
    )
    onset_evaluable = np.isfinite(ratios).all(axis=1)
    persist_evaluable = base.to_numpy() > 0
    evaluable = onset_evaluable & persist_evaluable

    onset = (ratios < onset_threshold).all(axis=1) & onset_evaluable
    persist_ratio = np.divide(
        pivot.loc[:, persist_months].mean(axis=1).to_numpy(float),
        base.to_numpy(float),
        out=np.full(len(base), np.nan, dtype=float),
        where=base.to_numpy(float) > 0,
    )
    persist = (persist_ratio < persist_threshold) & persist_evaluable
    weakened = evaluable & onset & persist

    onset_max_ratio = np.where(np.isfinite(ratios), ratios, -np.inf).max(axis=1)
    onset_max_ratio[~onset_evaluable] = np.nan

    return pd.DataFrame({
        'eligible': evaluable,
        'onset': onset,
        'persist': persist,
        'weakened': weakened,
        'onset_max_ratio': onset_max_ratio,
        'persist_ratio': persist_ratio,
    }, index=pivot.index)

def build_targets(cutoffs=MODEL_CUTOFFS, onset_threshold=ONSET_THRESHOLD, persist_threshold=PERSIST_THRESHOLD):
    all_rows = []
    for cutoff in cutoffs:
        parts = {
            axis: axis_weakening(AXIS_PIVOTS[axis], cutoff, onset_threshold, persist_threshold)
            for axis in AXIS_COLS
        }
        out = pd.DataFrame(index=FIRM_IDS)
        out.index.name = '법인ID'
        out['기준월'] = str(cutoff)
        for axis, part in parts.items():
            out[f'ELIGIBLE_{axis}'] = part['eligible'].to_numpy()
            out[f'ONSET_{axis}'] = part['onset'].to_numpy()
            out[f'PERSIST_{axis}'] = part['persist'].to_numpy()
            out[f'W_{axis}'] = part['weakened'].to_numpy()
            out[f'ONSET_MAX_RATIO_{axis}'] = part['onset_max_ratio'].to_numpy()
            out[f'PERSIST_RATIO_{axis}'] = part['persist_ratio'].to_numpy()

        out['TARGET_ELIGIBLE'] = out['ELIGIBLE_D'] & out[['ELIGIBLE_A', 'ELIGIBLE_C', 'ELIGIBLE_K']].any(axis=1)
        positive = out['W_D'] & out[['W_A', 'W_C', 'W_K']].any(axis=1)
        y = pd.Series(pd.NA, index=out.index, dtype='Int64')
        y.loc[out['TARGET_ELIGIBLE']] = positive.loc[out['TARGET_ELIGIBLE']].astype(int)
        out['Y_INTERVENE_M12_v2'] = y
        out['확인축조합'] = np.select(
            [
                out['W_A'] & out['W_C'] & out['W_K'],
                out['W_A'] & out['W_C'],
                out['W_A'] & out['W_K'],
                out['W_C'] & out['W_K'],
                out['W_A'], out['W_C'], out['W_K'],
            ],
            ['A+C+K', 'A+C', 'A+K', 'C+K', 'A', 'C', 'K'],
            default='',
        )
        all_rows.append(out.reset_index())
    return pd.concat(all_rows, ignore_index=True)

TARGETS = build_targets()
TARGETS.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_targets.csv', index=False, encoding='utf-8-sig')

target_by_cutoff = TARGETS.groupby('기준월', as_index=False).agg(
    전체=('법인ID', 'size'),
    적격=('TARGET_ELIGIBLE', 'sum'),
    양성=('Y_INTERVENE_M12_v2', 'sum'),
)
target_by_cutoff['양성률'] = target_by_cutoff['양성'] / target_by_cutoff['적격']
display(target_by_cutoff)

target_lock = {
    '전체격자': len(TARGETS),
    '적격': int(TARGETS['TARGET_ELIGIBLE'].sum()),
    '양성': int(TARGETS['Y_INTERVENE_M12_v2'].sum()),
    '양성률': float(TARGETS['Y_INTERVENE_M12_v2'].sum() / TARGETS['TARGET_ELIGIBLE'].sum()),
    '적격법인': int(TARGETS.loc[TARGETS['TARGET_ELIGIBLE'], '법인ID'].nunique()),
    '양성법인': int(TARGETS.loc[TARGETS['Y_INTERVENE_M12_v2'] == 1, '법인ID'].nunique()),
}
display(pd.Series(target_lock).to_frame('실행값'))

expected_target_lock = {'전체격자': 64068, '적격': 63572, '양성': 1966, '적격법인': 3354, '양성법인': 639}
if STRICT_TARGET_LOCK:
    for key, value in expected_target_lock.items():
        assert target_lock[key] == value, f'Y 잠금값 불일치: {key}={target_lock[key]} (기대 {value})'

target_by_cutoff.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_target_by_cutoff.csv', index=False, encoding='utf-8-sig')
TARGETS.loc[TARGETS['Y_INTERVENE_M12_v2'] == 1, '확인축조합'].value_counts().rename_axis('확인축조합').reset_index(name='양성행').to_csv(
    OUTPUT_DIR / 'model_m12_intervene_v2_positive_patterns.csv', index=False, encoding='utf-8-sig'
)


,기준월,전체,적격,양성,양성률
0,2023-12,3372,3348,85,0.025388
1,2024-01,3372,3347,91,0.027189
2,2024-02,3372,3344,94,0.02811
3,2024-03,3372,3344,85,0.025419
4,2024-04,3372,3346,98,0.029289
5,2024-05,3372,3346,100,0.029886
6,2024-06,3372,3344,77,0.023026
7,2024-07,3372,3343,82,0.024529
8,2024-08,3372,3344,94,0.02811
9,2024-09,3372,3346,106,0.03168


,실행값
전체격자,64068.000000
적격,63572.000000
양성,1966.000000
양성률,0.030926
적격법인,3354.000000
양성법인,639.000000


## 4. 롤링 피처 56개 생성

최종 입력은 다음 두 블록이다.

- R1 관계피처 16개: 수준 3, 활성폭 4, 구성 4, 순서형 강도 4, 카드 활성률 1
- D/A/C/K 동적피처 40개: 축별 10개

모든 피처는 법인×기준월별 `c-11~c`에서만 계산한다.


In [7]:
R1_FEATURES = [
    '핵심거래_수준', '수신자산_수준', '여신관계_수준',
    '수신상품_활성폭', '여신세부상품_활성폭', '채널_활성폭', '관계영역_활성폭',
    '수신_요구불대기타_로그균형', '여신_운전대시설_로그균형',
    '핵심거래_입출금대채널_평균비중', '채널_비대면대창구_로그균형',
    '수신좌수_강도', '여신부모좌수_강도', '신용카드개수_강도', '자동이체건수_강도',
    '카드_활성률',
]

AXIS_KO = {'D': '요구불', 'A': '자동이체', 'C': '채널', 'K': '카드'}
DYNAMIC_SUFFIXES = [
    '최근3대직전3_로그차이', '최근3대이전9_로그차이', 'H2대H1_로그차이',
    '로그변동성', 'TheilSen_추세', '최근3대이전9_활성률차이',
    '월간50퍼감소횟수', '월간감소율', '최대연속비활성개월', '현재월대직전6_로그차이',
]
DYNAMIC_FEATURES = [f'{AXIS_KO[a]}_{suffix}' for a in AXIS_COLS for suffix in DYNAMIC_SUFFIXES]
FS2_FEATURES = R1_FEATURES + DYNAMIC_FEATURES
assert len(R1_FEATURES) == 16 and len(DYNAMIC_FEATURES) == 40 and len(FS2_FEATURES) == 56

def build_r1_features(window):
    g = window.groupby('법인ID', sort=True)
    out = pd.DataFrame(index=FIRM_IDS)
    out.index.name = '법인ID'
    out['핵심거래_수준'] = g['핵심거래활동합계'].apply(lambda s: np.log1p(s).mean())
    out['수신자산_수준'] = g['수신자산합계'].apply(lambda s: np.log1p(s).mean())
    out['여신관계_수준'] = g['여신부모잔액합계'].apply(lambda s: np.log1p(s).mean())
    out['수신상품_활성폭'] = g['수신상품폭'].mean()
    out['여신세부상품_활성폭'] = g['여신세부상품폭'].mean()
    out['채널_활성폭'] = g['채널폭'].mean()
    out['관계영역_활성폭'] = g['관계영역폭'].mean()
    out['수신_요구불대기타_로그균형'] = g['요구불대기타수신_로그균형_월'].mean()
    out['여신_운전대시설_로그균형'] = g['운전대시설_로그균형_월'].mean()
    out['핵심거래_입출금대채널_평균비중'] = g['입출금대채널_비중_월'].mean()
    out['채널_비대면대창구_로그균형'] = g['비대면대창구_로그균형_월'].mean()
    out['수신좌수_강도'] = g['수신좌수_rank_월'].mean()
    out['여신부모좌수_강도'] = g['여신부모좌수_rank_월'].mean()
    out['신용카드개수_강도'] = g['신용카드개수_rank_월'].mean()
    out['자동이체건수_강도'] = g['자동이체건수_rank_월'].mean()
    out['카드_활성률'] = g['카드활성_월'].mean()
    return out[R1_FEATURES]

def build_axis_dynamic(axis, cutoff):
    months = pd.period_range(cutoff - 11, cutoff, freq='M')
    values = AXIS_PIVOTS[axis].loc[:, months].to_numpy(float)
    recent3, prev3, prev9 = values[:, 9:12], values[:, 6:9], values[:, :9]
    h1, h2 = values[:, :6], values[:, 6:12]
    previous6 = values[:, 5:11]
    prefix = AXIS_KO[axis]

    current = values[:, 1:]
    previous = values[:, :-1]
    valid_prev = previous > 0
    drop50 = ((current / np.where(valid_prev, previous, np.nan)) < 0.5)

    out = pd.DataFrame(index=FIRM_IDS)
    out[f'{prefix}_최근3대직전3_로그차이'] = logdiff(recent3.mean(1), prev3.mean(1))
    out[f'{prefix}_최근3대이전9_로그차이'] = logdiff(recent3.mean(1), prev9.mean(1))
    out[f'{prefix}_H2대H1_로그차이'] = logdiff(h2.mean(1), h1.mean(1))
    out[f'{prefix}_로그변동성'] = np.log1p(values).std(axis=1, ddof=0)
    out[f'{prefix}_TheilSen_추세'] = theil_sen_slopes(values)
    out[f'{prefix}_최근3대이전9_활성률차이'] = (recent3 > 0).mean(1) - (prev9 > 0).mean(1)
    out[f'{prefix}_월간50퍼감소횟수'] = np.nansum(drop50, axis=1)
    out[f'{prefix}_월간감소율'] = (current < previous).mean(axis=1)
    out[f'{prefix}_최대연속비활성개월'] = max_zero_run(values)
    out[f'{prefix}_현재월대직전6_로그차이'] = logdiff(values[:, -1], previous6.mean(1))
    return out

def build_feature_snapshot(cutoff):
    months = pd.period_range(cutoff - 11, cutoff, freq='M')
    window = MONTHLY[MONTHLY['월'].isin(months)]
    month_counts = window.groupby('법인ID')['월'].nunique().reindex(FIRM_IDS)
    assert month_counts.eq(12).all(), f'{cutoff}: 12개월 창이 완전하지 않습니다.'

    out = build_r1_features(window)
    for axis in AXIS_COLS:
        out = out.join(build_axis_dynamic(axis, cutoff), how='left')
    out = out.reset_index()
    out['기준월'] = str(cutoff)

    month_meta = MONTHLY.loc[MONTHLY['월'].eq(cutoff), [
        '법인ID', '업종_대분류', '업종_중분류', '사업장_시도', '사업장_시군구', '법인_고객등급', '전담고객여부'
    ]]
    out = out.merge(month_meta, on='법인ID', how='left', validate='one_to_one')
    return out

FEATURE_CUTOFFS = list(MODEL_CUTOFFS) + [OPERATING_CUTOFF]
FEATURE_ROWS = pd.concat([build_feature_snapshot(c) for c in FEATURE_CUTOFFS], ignore_index=True)

assert np.isfinite(FEATURE_ROWS[FS2_FEATURES].to_numpy(float)).all()
assert not FEATURE_ROWS[FS2_FEATURES].isna().any().any()
print('롤링 피처:', FEATURE_ROWS.shape, '| 최종 피처:', len(FS2_FEATURES))
display(FEATURE_ROWS[FS2_FEATURES].describe().T[['mean', 'std', 'min', 'max']].head(20))

FEATURE_ROWS.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_rolling_features.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({
    'feature': FS2_FEATURES,
    'block': ['R1_BASE16'] * 16 + ['DACK_DYNAMIC40'] * 40,
    'window': 'c-11~c',
    'available_as_of': 'cutoff_month_end',
}).to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_feature_registry.csv', index=False, encoding='utf-8-sig')


롤링 피처: (67440, 64) | 최종 피처: 56


,mean,std,min,max
핵심거래_수준,5.528135,2.076600,0.000000,14.187581
수신자산_수준,3.442537,2.320852,0.000000,12.711277
여신관계_수준,5.600017,2.309933,0.000000,12.033090
수신상품_활성폭,1.582716,1.041326,0.000000,5.083333
여신세부상품_활성폭,1.274603,0.772590,0.000000,6.000000
채널_활성폭,1.955624,1.214046,0.000000,5.000000
관계영역_활성폭,5.473499,0.737382,0.000000,6.000000
수신_요구불대기타_로그균형,0.973720,2.586553,-11.471567,8.580586
여신_운전대시설_로그균형,2.506748,4.305183,-11.373675,11.264353
핵심거래_입출금대채널_평균비중,0.723992,0.120571,0.277093,1.000000


## 5. 2023 고정 관계 세그먼트

2023년의 경험 백분위, 표준화값, 에너지 가중치와 `k=3` 중심을 잠근다. 이후 기준월에는 KMeans를 다시 적합하지 않고 같은 참조분포와 중심으로 현재 세그먼트를 배정한다.

세그먼트는 최종 X에서 제외한다.


In [8]:
try:
    from sklearn.cluster import KMeans
    from sklearn.isotonic import IsotonicRegression
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import (
        average_precision_score, roc_auc_score, brier_score_loss,
        log_loss, precision_recall_curve,
    )
    from lightgbm import LGBMClassifier
    import joblib
except Exception as exc:
    raise ImportError(
        '모델링 패키지 로드에 실패했습니다. 현재 커널에서 다음 셀을 별도로 실행한 뒤 커널을 재시작하세요:\n'
        '%pip install --upgrade --force-reinstall "numpy<2" "scipy>=1.11,<1.14" '
        '"scikit-learn>=1.4,<1.6" "lightgbm>=4.3,<5" "joblib>=1.3"'
    ) from exc

print('scikit-learn / LightGBM import 완료')


scikit-learn / LightGBM import 완료


In [9]:
R1_ENERGY_PCT = {
    '핵심거래_수준': 20.000, '수신자산_수준': 20.000, '여신관계_수준': 20.000,
    '수신상품_활성폭': 2.652, '여신세부상품_활성폭': 2.652,
    '채널_활성폭': 3.249, '관계영역_활성폭': 3.249,
    '수신_요구불대기타_로그균형': 2.652, '여신_운전대시설_로그균형': 2.652,
    '핵심거래_입출금대채널_평균비중': 3.249, '채널_비대면대창구_로그균형': 3.249,
    '수신좌수_강도': 2.652, '여신부모좌수_강도': 2.652,
    '신용카드개수_강도': 3.249, '자동이체건수_강도': 4.594, '카드_활성률': 3.249,
}
energy_sum = sum(R1_ENERGY_PCT.values())
R1_ENERGY_PCT = {k: v * (100.0 / energy_sum) for k, v in R1_ENERGY_PCT.items()}
R1_WEIGHTS = np.array([math.sqrt(R1_ENERGY_PCT[f] / 100.0) for f in R1_FEATURES])

reference = FEATURE_ROWS.query("기준월 == '2023-12'").set_index('법인ID').reindex(FIRM_IDS)
reference_pct = reference[R1_FEATURES].rank(method='average', pct=True)
reference_mean = reference_pct.mean(axis=0)
reference_std = reference_pct.std(axis=0, ddof=0).replace(0, 1.0)
reference_sorted = {f: np.sort(reference[f].to_numpy(float)) for f in R1_FEATURES}
X_segment_ref = ((reference_pct - reference_mean) / reference_std).to_numpy(float) * R1_WEIGHTS

segment_model = KMeans(
    n_clusters=3, init='k-means++', n_init=50, max_iter=500,
    algorithm='lloyd', tol=0.0001, random_state=RANDOM_STATE_SEGMENT,
)
reference_cluster = segment_model.fit_predict(X_segment_ref)

profile = reference_pct.assign(__cluster=reference_cluster).groupby('__cluster')[[
    '핵심거래_수준', '수신자산_수준', '여신관계_수준'
]].mean()
low_cluster = (profile['핵심거래_수준'] + profile['수신자산_수준']).idxmin()
remaining = [c for c in profile.index if c != low_cluster]
composite_cluster = profile.loc[remaining, '여신관계_수준'].idxmax()
middle_cluster = [c for c in profile.index if c not in [low_cluster, composite_cluster]][0]
SEGMENT_NAME = {
    int(composite_cluster): '복합고관계형',
    int(middle_cluster): '거래·수신중심형',
    int(low_cluster): '저거래·저수신형',
}
display(profile.assign(세그먼트=[SEGMENT_NAME[int(i)] for i in profile.index]))

def transform_segment_space(frame):
    pct = pd.DataFrame(index=frame.index)
    for f in R1_FEATURES:
        pct[f] = empirical_percentile(frame[f].to_numpy(float), reference_sorted[f])
    z = (pct - reference_mean) / reference_std
    return z.to_numpy(float) * R1_WEIGHTS

segment_rows = []
for cutoff, part in FEATURE_ROWS.groupby('기준월', sort=True):
    part = part.set_index('법인ID').reindex(FIRM_IDS)
    Xs = transform_segment_space(part)
    cluster = segment_model.predict(Xs)
    distances = np.linalg.norm(Xs[:, None, :] - segment_model.cluster_centers_[None, :, :], axis=2)
    ordered = np.sort(distances, axis=1)
    margin = (ordered[:, 1] - ordered[:, 0]) / np.maximum(ordered[:, 1], 1e-12)
    seg = pd.DataFrame({
        '법인ID': FIRM_IDS,
        '기준월': cutoff,
        'current_cluster': cluster,
        'current_segment': [SEGMENT_NAME[int(x)] for x in cluster],
        'segment_margin': margin,
        'segment_confidence': pd.cut(margin, [-np.inf, 0.10, 0.20, np.inf], labels=['경계형', '전이형', '명확형'], right=False).astype(str),
    })
    segment_rows.append(seg)

SEGMENTS = pd.concat(segment_rows, ignore_index=True)
baseline = SEGMENTS.query("기준월 == '2023-12'")[['법인ID', 'current_segment']].rename(columns={'current_segment': 'baseline_segment_2023'})
SEGMENTS = SEGMENTS.merge(baseline, on='법인ID', how='left', validate='many_to_one')
SEGMENTS['segment_transition'] = SEGMENTS['baseline_segment_2023'] + ' → ' + SEGMENTS['current_segment']

display(SEGMENTS.groupby(['기준월', 'current_segment']).size().unstack(fill_value=0).tail())
SEGMENTS.to_csv(OUTPUT_DIR / 'segmentation_current_assignment_by_firm.csv', index=False, encoding='utf-8-sig')
reference.reset_index().assign(
    baseline_cluster=reference_cluster,
    baseline_segment_2023=[SEGMENT_NAME[int(x)] for x in reference_cluster],
).to_csv(OUTPUT_DIR / 'final_segmentation_activity_revalidated_kmeans_k3.csv', index=False, encoding='utf-8-sig')


,핵심거래_수준,수신자산_수준,여신관계_수준,세그먼트
__cluster,,,,
0,0.208357,0.245193,0.458649,저거래·저수신형
1,0.700292,0.685277,0.762013,복합고관계형
2,0.634684,0.604514,0.219053,거래·수신중심형


current_segment,거래·수신중심형,복합고관계형,저거래·저수신형
기준월,,,
2025-03,935,1218,1219
2025-04,935,1212,1225
2025-05,938,1205,1229
2025-06,942,1203,1227
2025-12,934,1181,1257


## 6. 모델 데이터와 Purged Walk-Forward 분할

Y가 기준월 이후 6개월을 사용하므로 Validation 직전 5개 기준월을 학습에서 제외한다. 최종 Test는 2025.06 한 번만 사용한다.


In [10]:
eligible_targets = TARGETS.loc[TARGETS['TARGET_ELIGIBLE']].copy()
eligible_targets['Y_INTERVENE_M12_v2'] = eligible_targets['Y_INTERVENE_M12_v2'].astype(int)

MODEL_DATA = eligible_targets.merge(FEATURE_ROWS, on=['법인ID', '기준월'], how='left', validate='one_to_one')
MODEL_DATA = MODEL_DATA.merge(
    SEGMENTS[['법인ID', '기준월', 'baseline_segment_2023', 'current_segment', 'segment_transition', 'segment_margin', 'segment_confidence']],
    on=['법인ID', '기준월'], how='left', validate='one_to_one',
)
assert len(MODEL_DATA) == 63572
assert not MODEL_DATA[FS2_FEATURES].isna().any().any()
MODEL_DATA.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_dataset.csv', index=False, encoding='utf-8-sig')

FOLDS = [
    {'fold': 1, 'train_start': '2023-12', 'train_end': '2024-04', 'purge_start': '2024-05', 'purge_end': '2024-09', 'validation': '2024-10'},
    {'fold': 2, 'train_start': '2023-12', 'train_end': '2024-05', 'purge_start': '2024-06', 'purge_end': '2024-10', 'validation': '2024-11'},
    {'fold': 3, 'train_start': '2023-12', 'train_end': '2024-06', 'purge_start': '2024-07', 'purge_end': '2024-11', 'validation': '2024-12'},
]

split_rows = []
for spec in FOLDS:
    train_mask = MODEL_DATA['기준월'].between(spec['train_start'], spec['train_end'])
    val_mask = MODEL_DATA['기준월'].eq(spec['validation'])
    split_rows.append({
        **spec,
        'train_rows': int(train_mask.sum()),
        'train_positive': int(MODEL_DATA.loc[train_mask, 'Y_INTERVENE_M12_v2'].sum()),
        'validation_rows': int(val_mask.sum()),
        'validation_positive': int(MODEL_DATA.loc[val_mask, 'Y_INTERVENE_M12_v2'].sum()),
        'train_last_y_month': str(pd.Period(spec['train_end'], freq='M') + 6),
        'validation_first_y_month': str(pd.Period(spec['validation'], freq='M') + 1),
    })

split_registry = pd.DataFrame(split_rows)
split_registry['no_y_window_overlap'] = pd.PeriodIndex(split_registry['train_last_y_month'], freq='M') < pd.PeriodIndex(split_registry['validation_first_y_month'], freq='M')
assert split_registry['no_y_window_overlap'].all()
display(split_registry)
split_registry.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_split_registry.csv', index=False, encoding='utf-8-sig')

expected_fold_rows = [(16729, 3348), (20075, 3349), (23419, 3350)]
for row, expected in zip(split_rows, expected_fold_rows):
    assert (row['train_rows'], row['validation_rows']) == expected


,fold,train_start,train_end,purge_start,purge_end,validation,train_rows,train_positive,validation_rows,validation_positive,train_last_y_month,validation_first_y_month,no_y_window_overlap
0,1,2023-12,2024-04,2024-05,2024-09,2024-10,16729,453,3348,113,2024-10,2024-11,True
1,2,2023-12,2024-05,2024-06,2024-10,2024-11,20075,553,3349,107,2024-11,2024-12,True
2,3,2023-12,2024-06,2024-07,2024-11,2024-12,23419,630,3350,111,2024-12,2025-01,True


## 7. LightGBM Walk-Forward와 최종 Test

최종 잠금 피처는 56개, 모델은 LightGBM, 확률 보정은 Train 법인 그룹 OOF로 선택한 Isotonic이다. 순위 동점은 `보정확률 내림차순 → 법인ID 오름차순`으로 고정한다.


In [11]:
LOCKED_LGBM_PARAMS = {
    'n_estimators': 220,
    'learning_rate': 0.024068490013144556,
    'num_leaves': 22,
    'max_depth': 6,
    'min_child_samples': 85,
    'subsample': 0.9978564144159624,
    'colsample_bytree': 0.6264619236335867,
    'reg_alpha': 0.001889859045062956,
    'reg_lambda': 9.394600689281587,
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE_MODEL,
    'n_jobs': -1,
    'verbosity': -1,
}

def new_model():
    return LGBMClassifier(**LOCKED_LGBM_PARAMS)

def deterministic_order(ids, scores):
    return np.lexsort((np.asarray(ids).astype(str), -np.asarray(scores, dtype=float)))

def topk_metrics(y_true, scores, ids, fraction=0.10):
    y_true = np.asarray(y_true, dtype=int)
    order = deterministic_order(ids, scores)
    n_top = int(math.ceil(len(y_true) * fraction))
    selected = order[:n_top]
    positives = y_true.sum()
    captured = y_true[selected].sum()
    recall = captured / positives if positives else np.nan
    precision = captured / n_top if n_top else np.nan
    prevalence = positives / len(y_true) if len(y_true) else np.nan
    lift = precision / prevalence if prevalence else np.nan
    return {'n_top': n_top, 'captured': int(captured), 'recall': recall, 'precision': precision, 'lift': lift}

def ece_score(y_true, prob, bins=10):
    y_true = np.asarray(y_true, dtype=float)
    prob = np.asarray(prob, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(prob, edges[1:-1]), 0, bins - 1)
    ece = 0.0
    for b in range(bins):
        mask = idx == b
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)

def metric_row(y_true, prob, ids, label):
    top5 = topk_metrics(y_true, prob, ids, 0.05)
    top10 = topk_metrics(y_true, prob, ids, 0.10)
    top20 = topk_metrics(y_true, prob, ids, 0.20)
    return {
        'label': label,
        'rows': len(y_true),
        'positive': int(np.sum(y_true)),
        'pr_auc': average_precision_score(y_true, prob),
        'roc_auc': roc_auc_score(y_true, prob),
        'recall_at_5pct': top5['recall'],
        'recall_at_10pct': top10['recall'],
        'precision_at_10pct': top10['precision'],
        'lift_at_10pct': top10['lift'],
        'recall_at_20pct': top20['recall'],
        'brier': brier_score_loss(y_true, prob),
        'ece10': ece_score(y_true, prob, 10),
    }

walk_rows, walk_predictions = [], []
for spec in FOLDS:
    train = MODEL_DATA[MODEL_DATA['기준월'].between(spec['train_start'], spec['train_end'])]
    valid = MODEL_DATA[MODEL_DATA['기준월'].eq(spec['validation'])]
    model = new_model()
    model.fit(train[FS2_FEATURES], train['Y_INTERVENE_M12_v2'])
    prob = model.predict_proba(valid[FS2_FEATURES])[:, 1]
    row = metric_row(valid['Y_INTERVENE_M12_v2'], prob, valid['법인ID'], f"fold_{spec['fold']}")
    row['fold'] = spec['fold']
    walk_rows.append(row)
    walk_predictions.append(valid[['법인ID', '기준월', 'Y_INTERVENE_M12_v2']].assign(raw_probability=prob, fold=spec['fold']))

WALK_METRICS = pd.DataFrame(walk_rows)
WALK_PREDICTIONS = pd.concat(walk_predictions, ignore_index=True)
display(WALK_METRICS)
WALK_METRICS.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_walk_forward_metrics.csv', index=False, encoding='utf-8-sig')
WALK_PREDICTIONS.to_csv(OUTPUT_DIR / 'model_validation_oof_predictions.csv', index=False, encoding='utf-8-sig')


,label,rows,positive,pr_auc,roc_auc,recall_at_5pct,recall_at_10pct,precision_at_10pct,lift_at_10pct,recall_at_20pct,brier,ece10,fold
0,fold_1,3348,113,0.353399,0.898587,0.486726,0.654867,0.220896,6.544763,0.805310,0.078259,0.136966,1
1,fold_2,3349,107,0.303305,0.907963,0.457944,0.616822,0.197015,6.166383,0.850467,0.091939,0.165616,2
2,fold_3,3350,111,0.331797,0.904762,0.468468,0.702703,0.232836,7.027027,0.873874,0.073922,0.141632,3


In [12]:
FINAL_TRAIN = MODEL_DATA[MODEL_DATA['기준월'].between('2023-12', '2024-12')].copy()
FINAL_TEST = MODEL_DATA[MODEL_DATA['기준월'].eq('2025-06')].copy()
assert len(FINAL_TRAIN) == 43499 and len(FINAL_TEST) == 3346
assert FINAL_TRAIN['Y_INTERVENE_M12_v2'].sum() == 1243 and FINAL_TEST['Y_INTERVENE_M12_v2'].sum() == 119

def grouped_oof_raw(data, n_splits=5):
    y = data['Y_INTERVENE_M12_v2'].to_numpy(int)
    groups = data['법인ID'].astype(str).to_numpy()
    oof = np.full(len(data), np.nan, dtype=float)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE_MODEL)
    for train_idx, val_idx in splitter.split(data[FS2_FEATURES], y, groups):
        model = new_model()
        model.fit(data.iloc[train_idx][FS2_FEATURES], y[train_idx])
        oof[val_idx] = model.predict_proba(data.iloc[val_idx][FS2_FEATURES])[:, 1]
    assert np.isfinite(oof).all()
    return oof

final_train_oof_raw = grouped_oof_raw(FINAL_TRAIN, n_splits=5)
final_calibrator = IsotonicRegression(out_of_bounds='clip')
final_calibrator.fit(final_train_oof_raw, FINAL_TRAIN['Y_INTERVENE_M12_v2'].to_numpy(int))

final_model = new_model()
final_model.fit(FINAL_TRAIN[FS2_FEATURES], FINAL_TRAIN['Y_INTERVENE_M12_v2'])
test_raw = final_model.predict_proba(FINAL_TEST[FS2_FEATURES])[:, 1]
test_probability = final_calibrator.predict(test_raw)

TEST_PREDICTIONS = FINAL_TEST[[
    '법인ID', '기준월', 'Y_INTERVENE_M12_v2', '업종_대분류', '업종_중분류',
    'baseline_segment_2023', 'current_segment', 'segment_transition'
]].copy()
TEST_PREDICTIONS['raw_probability'] = test_raw
TEST_PREDICTIONS['risk_probability'] = test_probability
order = deterministic_order(TEST_PREDICTIONS['법인ID'], TEST_PREDICTIONS['risk_probability'])
rank = np.empty(len(TEST_PREDICTIONS), dtype=int)
rank[order] = np.arange(1, len(TEST_PREDICTIONS) + 1)
TEST_PREDICTIONS['risk_rank'] = rank
TEST_PREDICTIONS['top10_flag'] = TEST_PREDICTIONS['risk_rank'] <= math.ceil(len(TEST_PREDICTIONS) * 0.10)

TEST_METRICS = pd.DataFrame([metric_row(
    FINAL_TEST['Y_INTERVENE_M12_v2'], test_probability, FINAL_TEST['법인ID'], 'LightGBM_Isotonic_2025-06'
)])
display(TEST_METRICS.T)

TEST_PREDICTIONS.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_final_test_predictions.csv', index=False, encoding='utf-8-sig')
TEST_METRICS.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_final_test_metrics.csv', index=False, encoding='utf-8-sig')
joblib.dump(final_model, MODEL_DIR / 'model_m12_intervene_v2_final_test_pipeline.joblib')
joblib.dump(final_calibrator, MODEL_DIR / 'model_m12_intervene_v2_final_test_calibrator.joblib')


,0
label,LightGBM_Isotonic_2025-06
rows,3346
positive,119
pr_auc,0.327375
roc_auc,0.896855
recall_at_5pct,0.445378
recall_at_10pct,0.672269
precision_at_10pct,0.238806
lift_at_10pct,6.714662
recall_at_20pct,0.823529


['/Users/changeun_1/Desktop/iM/5. final/수익성F_models/model_m12_intervene_v2_final_test_calibrator.joblib']

## 8. Test 부트스트랩, 집단 안정성, SHAP, 드리프트

세그먼트와 업종은 최종 모델에 넣지 않고 Test 결과의 집단별 안정성만 확인한다. SHAP은 LightGBM의 `pred_contrib`로 계산하며 인과효과가 아니라 예측 기여도다.


In [13]:
if RUN_BOOTSTRAP:
    rng = np.random.default_rng(RANDOM_STATE_MODEL)
    boot_rows = []
    y_test = FINAL_TEST['Y_INTERVENE_M12_v2'].to_numpy(int)
    ids_test = FINAL_TEST['법인ID'].astype(str).to_numpy()
    for b in range(BOOTSTRAP_REPS):
        idx = rng.integers(0, len(FINAL_TEST), len(FINAL_TEST))
        yb, pb, ib = y_test[idx], test_probability[idx], ids_test[idx]
        if yb.min() == yb.max():
            continue
        row = metric_row(yb, pb, ib, b)
        boot_rows.append(row)
    BOOTSTRAP = pd.DataFrame(boot_rows)
    summary_rows = []
    for metric in ['pr_auc', 'roc_auc', 'recall_at_10pct', 'precision_at_10pct', 'lift_at_10pct', 'brier']:
        summary_rows.append({
            'metric': metric,
            'median': BOOTSTRAP[metric].median(),
            'p2_5': BOOTSTRAP[metric].quantile(0.025),
            'p97_5': BOOTSTRAP[metric].quantile(0.975),
        })
    BOOTSTRAP_SUMMARY = pd.DataFrame(summary_rows)
    display(BOOTSTRAP_SUMMARY)
    BOOTSTRAP_SUMMARY.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_test_bootstrap_summary.csv', index=False, encoding='utf-8-sig')

def group_stability(predictions, group_col, min_rows=30):
    rows = []
    for group, part in predictions.groupby(group_col, dropna=False):
        if len(part) < min_rows or part['Y_INTERVENE_M12_v2'].nunique() < 2:
            continue
        selected = part['top10_flag']
        positive = part['Y_INTERVENE_M12_v2'].sum()
        rows.append({
            group_col: group,
            'rows': len(part),
            'positive': int(positive),
            'positive_rate': positive / len(part),
            'global_top10_selected': int(selected.sum()),
            'positive_capture_rate': part.loc[selected, 'Y_INTERVENE_M12_v2'].sum() / positive if positive else np.nan,
            'within_group_pr_auc': average_precision_score(part['Y_INTERVENE_M12_v2'], part['risk_probability']),
        })
    return pd.DataFrame(rows)

SEGMENT_STABILITY = group_stability(TEST_PREDICTIONS, 'current_segment')
INDUSTRY_STABILITY = group_stability(TEST_PREDICTIONS, '업종_대분류', min_rows=100).sort_values('rows', ascending=False)
display(SEGMENT_STABILITY)
display(INDUSTRY_STABILITY.head(10))
SEGMENT_STABILITY.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_test_segment_stability.csv', index=False, encoding='utf-8-sig')
INDUSTRY_STABILITY.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_test_industry_stability.csv', index=False, encoding='utf-8-sig')

test_contrib = final_model.booster_.predict(FINAL_TEST[FS2_FEATURES], pred_contrib=True)
TEST_SHAP_IMPORTANCE = pd.DataFrame({
    'feature': FS2_FEATURES,
    'mean_abs_shap': np.abs(test_contrib[:, :-1]).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
display(TEST_SHAP_IMPORTANCE.head(20))
TEST_SHAP_IMPORTANCE.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_test_shap_importance.csv', index=False, encoding='utf-8-sig')

def psi(reference_values, comparison_values, bins=10, eps=1e-6):
    ref = np.asarray(reference_values, dtype=float)
    cmp = np.asarray(comparison_values, dtype=float)
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_hist = np.histogram(ref, bins=edges)[0] / len(ref)
    cmp_hist = np.histogram(cmp, bins=edges)[0] / len(cmp)
    ref_hist = np.clip(ref_hist, eps, None)
    cmp_hist = np.clip(cmp_hist, eps, None)
    return float(np.sum((cmp_hist - ref_hist) * np.log(cmp_hist / ref_hist)))

TEST_DRIFT = pd.DataFrame({
    'feature': FS2_FEATURES,
    'psi_train_to_test': [psi(FINAL_TRAIN[f], FINAL_TEST[f]) for f in FS2_FEATURES],
})
TEST_DRIFT['status'] = pd.cut(TEST_DRIFT['psi_train_to_test'], [-np.inf, 0.10, 0.25, np.inf], labels=['LOW', 'WATCH', 'HIGH'], right=False)
display(TEST_DRIFT.sort_values('psi_train_to_test', ascending=False).head(15))


,metric,median,p2_5,p97_5
0,pr_auc,0.331840,0.251836,0.415501
1,roc_auc,0.897139,0.866953,0.923196
2,recall_at_10pct,0.669565,0.592560,0.754907
3,precision_at_10pct,0.238806,0.191045,0.292612
4,lift_at_10pct,6.687657,5.918529,7.540056
5,brier,0.027897,0.023624,0.032462


,current_segment,rows,positive,positive_rate,global_top10_selected,positive_capture_rate,within_group_pr_auc
0,거래·수신중심형,942,24,0.025478,94,0.666667,0.262008
1,복합고관계형,1203,28,0.023275,94,0.750000,0.366979
2,저거래·저수신형,1201,67,0.055787,147,0.641791,0.375060


,업종_대분류,rows,positive,positive_rate,global_top10_selected,positive_capture_rate,within_group_pr_auc
5,제조업,1084,33,0.030443,100,0.606061,0.304655
1,도매 및 소매업,704,33,0.046875,57,0.636364,0.495209
0,건설업,580,22,0.037931,102,0.727273,0.220047
2,부동산업,264,8,0.030303,23,0.875000,0.359606
4,운수 및 창고업,142,7,0.049296,9,0.571429,0.539891
3,"사업시설 관리, 사업 지원 및 임대 서비스업",106,5,0.047170,9,0.600000,0.580586


,feature,mean_abs_shap
20,요구불_TheilSen_추세,0.458771
5,채널_활성폭,0.299817
40,채널_TheilSen_추세,0.281924
19,요구불_로그변동성,0.221759
30,자동이체_TheilSen_추세,0.149477
9,핵심거래_입출금대채널_평균비중,0.147586
0,핵심거래_수준,0.133059
17,요구불_최근3대이전9_로그차이,0.123863
39,채널_로그변동성,0.116232
2,여신관계_수준,0.100670


,feature,psi_train_to_test,status
50,카드_TheilSen_추세,0.080570,LOW
18,요구불_H2대H1_로그차이,0.053643,LOW
30,자동이체_TheilSen_추세,0.048723,LOW
53,카드_월간감소율,0.046569,LOW
28,자동이체_H2대H1_로그차이,0.041526,LOW
54,카드_최대연속비활성개월,0.039321,LOW
20,요구불_TheilSen_추세,0.038721,LOW
48,카드_H2대H1_로그차이,0.037961,LOW
27,자동이체_최근3대이전9_로그차이,0.036204,LOW
40,채널_TheilSen_추세,0.030891,LOW


## 9. 2025.12 운영 모델과 위험 리스트

Test 평가가 끝난 뒤 구조·피처·파라미터·보정 종류를 바꾸지 않고, 2025.06까지 Y가 확정된 전체 적격 데이터로 재학습한다. 2025.12 점수의 실제 Y는 2026.01~2026.06 자료가 있어야 평가할 수 있다.


In [14]:
OPERATING_FEATURES = FEATURE_ROWS.query("기준월 == '2025-12'").copy()
OPERATING_FEATURES = OPERATING_FEATURES.merge(
    SEGMENTS.query("기준월 == '2025-12'")[[
        '법인ID', '기준월', 'baseline_segment_2023', 'current_segment', 'segment_transition', 'segment_margin', 'segment_confidence'
    ]],
    on=['법인ID', '기준월'], how='left', validate='one_to_one',
)

op_months = pd.period_range(OPERATING_CUTOFF - 11, OPERATING_CUTOFF, freq='M')
op_yoy_denominator_months = pd.period_range(OPERATING_CUTOFF - 11, OPERATING_CUTOFF - 9, freq='M')
op_axis_evaluable = {}
for axis, pivot in AXIS_PIVOTS.items():
    base_positive = pivot.loc[:, op_months].mean(axis=1) > 0
    yoy_denominators_positive = (pivot.loc[:, op_yoy_denominator_months] > 0).all(axis=1)
    op_axis_evaluable[axis] = base_positive & yoy_denominators_positive
op_target_evaluable = op_axis_evaluable['D'] & pd.concat(
    [op_axis_evaluable['A'], op_axis_evaluable['C'], op_axis_evaluable['K']], axis=1
).any(axis=1)
op_eligible_ids = op_target_evaluable.index[op_target_evaluable]
OPERATING_FEATURES = OPERATING_FEATURES[OPERATING_FEATURES['법인ID'].isin(op_eligible_ids)].copy()
print('2025.12 전체 법인:', len(FIRM_IDS), '| 점수 적격:', len(OPERATING_FEATURES))
if STRICT_TARGET_LOCK:
    assert len(OPERATING_FEATURES) == 3341, f'운영 적격 잠금값 불일치: {len(OPERATING_FEATURES)}'

operating_oof_raw = grouped_oof_raw(MODEL_DATA, n_splits=5)
operating_calibrator = IsotonicRegression(out_of_bounds='clip')
operating_calibrator.fit(operating_oof_raw, MODEL_DATA['Y_INTERVENE_M12_v2'].to_numpy(int))

operating_model = new_model()
operating_model.fit(MODEL_DATA[FS2_FEATURES], MODEL_DATA['Y_INTERVENE_M12_v2'])
operating_raw = operating_model.predict_proba(OPERATING_FEATURES[FS2_FEATURES])[:, 1]
operating_probability = operating_calibrator.predict(operating_raw)

OPERATING_SCORES = OPERATING_FEATURES[[
    '법인ID', '기준월', '업종_대분류', '업종_중분류', '사업장_시도', '사업장_시군구',
    '법인_고객등급', '전담고객여부', 'baseline_segment_2023', 'current_segment',
    'segment_transition', 'segment_margin', 'segment_confidence',
]].copy()
OPERATING_SCORES['raw_probability'] = operating_raw
OPERATING_SCORES['risk_probability'] = operating_probability
order = deterministic_order(OPERATING_SCORES['법인ID'], OPERATING_SCORES['risk_probability'])
rank = np.empty(len(OPERATING_SCORES), dtype=int)
rank[order] = np.arange(1, len(OPERATING_SCORES) + 1)
OPERATING_SCORES['risk_rank'] = rank
OPERATING_SCORES['top10_flag'] = OPERATING_SCORES['risk_rank'] <= math.ceil(len(OPERATING_SCORES) * 0.10)

key_change_features = [
    '요구불_최근3대이전9_로그차이', '자동이체_최근3대이전9_로그차이',
    '채널_최근3대이전9_로그차이', '카드_최근3대이전9_로그차이',
]
OPERATING_SCORES = OPERATING_SCORES.merge(
    OPERATING_FEATURES[['법인ID'] + key_change_features], on='법인ID', how='left', validate='one_to_one'
)

op_contrib = operating_model.booster_.predict(OPERATING_FEATURES[FS2_FEATURES], pred_contrib=True)[:, :-1]
top_idx = np.argsort(-np.abs(op_contrib), axis=1)[:, :10]
for j in range(10):
    OPERATING_SCORES[f'top{j+1}_shap_feature'] = [FS2_FEATURES[k] for k in top_idx[:, j]]
    OPERATING_SCORES[f'top{j+1}_shap_value'] = op_contrib[np.arange(len(op_contrib)), top_idx[:, j]]

OPERATING_SCORES = OPERATING_SCORES.sort_values(['risk_rank', '법인ID']).reset_index(drop=True)
OPERATING_TOP10 = OPERATING_SCORES[OPERATING_SCORES['top10_flag']].copy()
display(OPERATING_SCORES.head(20))
display(OPERATING_SCORES['risk_probability'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

OPERATING_FEATURES.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_operating_features_202512.csv', index=False, encoding='utf-8-sig')
OPERATING_SCORES.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_operating_scores_202512.csv', index=False, encoding='utf-8-sig')
OPERATING_TOP10.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_operating_top10_202512.csv', index=False, encoding='utf-8-sig')
joblib.dump(operating_model, MODEL_DIR / 'model_m12_intervene_v2_operating_pipeline_202512.joblib')
joblib.dump(operating_calibrator, MODEL_DIR / 'model_m12_intervene_v2_operating_calibrator_202512.joblib')

OPERATING_DRIFT = pd.DataFrame({
    'feature': FS2_FEATURES,
    'psi_train_to_test': [psi(FINAL_TRAIN[f], FINAL_TEST[f]) for f in FS2_FEATURES],
    'psi_train_to_202512': [psi(FINAL_TRAIN[f], OPERATING_FEATURES[f]) for f in FS2_FEATURES],
})
OPERATING_DRIFT['status_202512'] = pd.cut(OPERATING_DRIFT['psi_train_to_202512'], [-np.inf, 0.10, 0.25, np.inf], labels=['LOW', 'WATCH', 'HIGH'], right=False)
OPERATING_DRIFT.to_csv(OUTPUT_DIR / 'model_m12_intervene_v2_feature_drift.csv', index=False, encoding='utf-8-sig')


2025.12 전체 법인: 3372 | 점수 적격: 3341


,법인ID,기준월,업종_대분류,업종_중분류,사업장_시도,사업장_시군구,법인_고객등급,전담고객여부,baseline_segment_2023,current_segment,segment_transition,segment_margin,segment_confidence,raw_probability,risk_probability,risk_rank,top10_flag,요구불_최근3대이전9_로그차이,자동이체_최근3대이전9_로그차이,채널_최근3대이전9_로그차이,카드_최근3대이전9_로그차이,top1_shap_feature,top1_shap_value,top2_shap_feature,top2_shap_value,top3_shap_feature,top3_shap_value
0,ae78ab5b2ff86031aab3d3f6438c81497cda9560754c55...,2025-12,도매 및 소매업,소매업; 자동차 제외,부울경,경상남도,우수,N,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,0.424638,명확형,0.972594,0.891173,1,True,-2.319695,-1.188049,-2.374126,0.000000,요구불_TheilSen_추세,2.301356,채널_TheilSen_추세,0.700248,자동이체_TheilSen_추세,0.496102
1,0a77628abb8c5baeb68b65a896daf9a38e27b90bdc4ec3...,2025-12,제조업,기타 기계 및 장비 제조업,부울경,경상남도,일반,Y,거래·수신중심형,저거래·저수신형,거래·수신중심형 → 저거래·저수신형,0.318418,명확형,0.970798,0.857143,2,True,-1.484628,-0.461073,-1.758260,-0.922494,요구불_TheilSen_추세,2.147800,채널_TheilSen_추세,0.641321,자동이체_TheilSen_추세,0.373240
2,be4c559f18493bab29c7f77bb1034e6dc884dd402509f8...,2025-12,도매 및 소매업,도매 및 상품 중개업,수도권,인천광역시,최우수,Y,복합고관계형,복합고관계형,복합고관계형 → 복합고관계형,0.328926,명확형,0.971303,0.857143,3,True,-2.174752,-0.958693,-3.011577,-1.356040,요구불_TheilSen_추세,2.245351,채널_TheilSen_추세,0.710228,자동이체_TheilSen_추세,0.356492
3,db697eb8f79eb71c22a76a9156429deba0cbd961ac1cef...,2025-12,제조업,"전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업",대구,대구광역시,우수,N,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,0.512756,명확형,0.970796,0.857143,4,True,-3.856980,-1.852994,-3.101992,-0.525386,요구불_TheilSen_추세,2.070363,채널_TheilSen_추세,0.648252,자동이체_TheilSen_추세,0.441787
4,0f4f0c4b9942d0bd7678d86cf9ea47fd44dbcf6d650edb...,2025-12,도매 및 소매업,도매 및 상품 중개업,부울경,경상남도,우수,Y,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,0.609129,명확형,0.964260,0.845070,5,True,-1.915465,-0.227071,-3.897473,-0.095310,요구불_TheilSen_추세,2.580637,채널_TheilSen_추세,0.804445,자동이체_TheilSen_추세,0.426873
5,409a17ccf9aa04a06cec3294ace29bb542b9b2f574b6a7...,2025-12,건설업,전문직별 공사업,대구,대구광역시,일반,Y,거래·수신중심형,저거래·저수신형,거래·수신중심형 → 저거래·저수신형,0.583061,명확형,0.964321,0.845070,6,True,-3.145681,-1.095739,-2.900456,-0.740730,요구불_TheilSen_추세,2.221715,채널_TheilSen_추세,0.685502,자동이체_TheilSen_추세,0.401358
6,489493424e4301baefd4bc90914c72b848d3e95b746f1a...,2025-12,도매 및 소매업,소매업; 자동차 제외,NaN,NaN,우수,Y,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,0.529861,명확형,0.964258,0.845070,7,True,-4.509503,-1.271631,-3.879375,-1.405552,요구불_TheilSen_추세,2.258486,채널_TheilSen_추세,0.646049,자동이체_TheilSen_추세,0.392264
7,685d3200b2e5de34133fe93eae056e3c85e1e0f34bbee7...,2025-12,제조업,목재 및 나무제품 제조업; 가구 제외,경북,경상북도,우수,Y,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,0.451774,명확형,0.969980,0.845070,8,True,-2.833556,-0.612923,-3.695813,0.064444,요구불_TheilSen_추세,2.335892,채널_TheilSen_추세,0.746334,자동이체_TheilSen_추세,0.412510
8,722835e512acd998274d62e431d24188dc147da85f0b2b...,2025-12,"수도, 하수 및 폐기물 처리, 원료 재생업","폐기물 수집, 운반, 처리 및 원료 재생업",대구,대구광역시,최우수,Y,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,0.355272,명확형,0.964632,0.845070,9,True,-2.153874,-2.521862,-1.812643,-1.151275,요구불_TheilSen_추세,2.196500,채널_TheilSen_추세,0.662689,자동이체_TheilSen_추세,0.565589
9,96bbb1c9e6f91c91639f7939169932234bfa7f03a66f38...,2025-12,도매 및 소매업,소매업; 자동차 제외,대구,대구광역시,일반,N,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,0.456464,명확형,0.966944,0.845070,10,True,-2.362256,-0.888491,-2.669164,0.000000,요구불_TheilSen_추세,2.416197,채널_TheilSen_추세,0.764999,자동이체_TheilSen_추세,0.493881


count    3341.000000
mean        0.034610
std         0.093822
min         0.000000
50%         0.006753
90%         0.073363
95%         0.178571
99%         0.480263
max         0.891173
Name: risk_probability, dtype: float64

## 10. 최종 재현성 잠금과 수익성 연결 준비

이 노트북에서 만들어지는 `risk_probability`, `risk_rank`, `top10_flag`는 다음 수익성 단계의 위험축이다. 세그먼트는 확률을 조정하지 않고 관계 맥락만 제공한다.

수익성 단계에서는 같은 `법인ID`와 기준월로 FISIM 기반 기간누적 고객가치와 결합한다. 이때 위험확률로 가치를 나누지 않고, 필요하면 `risk_probability × max(LTV*, 0)` 형태의 위험노출가치를 별도로 계산한다.


In [15]:
RUN_SUMMARY = {
    'source_path': str(SOURCE_PATH),
    'source_sha256': sha256_file(SOURCE_PATH),
    'target_rows': len(TARGETS),
    'eligible_rows': int(TARGETS['TARGET_ELIGIBLE'].sum()),
    'positive_rows': int(TARGETS['Y_INTERVENE_M12_v2'].sum()),
    'model_rows': len(MODEL_DATA),
    'feature_count': len(FS2_FEATURES),
    'final_train_rows': len(FINAL_TRAIN),
    'final_test_rows': len(FINAL_TEST),
    'final_test_positive': int(FINAL_TEST['Y_INTERVENE_M12_v2'].sum()),
    'test_metrics': TEST_METRICS.iloc[0].to_dict(),
    'operating_cutoff': str(OPERATING_CUTOFF),
    'operating_scored': len(OPERATING_SCORES),
    'operating_top10': len(OPERATING_TOP10),
    'segment_used_in_final_x': False,
    'final_feature_set': 'FS2_R1_DACK_DYNAMIC',
    'final_model': 'LightGBM + Isotonic',
}
with open(OUTPUT_DIR / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(RUN_SUMMARY, f, ensure_ascii=False, indent=2, default=str)

display(pd.Series({
    '적격 모델행': len(MODEL_DATA),
    '양성 모델행': int(MODEL_DATA['Y_INTERVENE_M12_v2'].sum()),
    '최종 피처': len(FS2_FEATURES),
    'Test 행': len(FINAL_TEST),
    'Test 양성': int(FINAL_TEST['Y_INTERVENE_M12_v2'].sum()),
    '운영 점수 법인': len(OPERATING_SCORES),
    '운영 상위10%': len(OPERATING_TOP10),
}).to_frame('최종 실행 결과'))

print('완료:', OUTPUT_DIR)


,최종 실행 결과
적격 모델행,63572
양성 모델행,1966
최종 피처,56
Test 행,3346
Test 양성,119
운영 점수 법인,3341
운영 상위10%,335


완료: /Users/changeun_1/Desktop/iM/5. final/수익성F_outputs


## 11. 수익성 지표와 월별 FISIM 경제적 기여가치

수정본 「수익성 지표 및 고객생애가치(CLV) 산정」의 정의를 그대로 구현한다.

\[
V^{FTP}_{i,m}=L_{i,m}(r^L_m-FTP_m)+D^S_{i,m}(FTP_m-r^S_m)+D^R_{i,m}(FTP_m-r^R)
\]

- `L`: 운전자금대출잔액 + 시설자금대출잔액
- `DS`: 거치식예금잔액 + 적립식예금잔액
- `DR`: 요구불예금잔액
- 기업대출·저축성수신 금리 CSV는 **월 기준 %**이므로 `/100`만 적용
- 추정 FTP CSV는 **월 기준 소수**이므로 그대로 적용
- 요구불금리는 수정본의 `0.01% = 0.0001` 월 소수 가정을 적용
- 원천 월 FISIM의 음수는 역마진 진단을 위해 보존하고, 방어순위에서만 잠재손실에 0 하한을 둔다.

자동이체·채널·카드·관계피처·세그먼트는 금액 산식에 직접 넣지 않는다. 세그먼트는 계산이 끝난 수익성·CLV를 비교하는 설명 정보로만 사용한다.

In [16]:
import re
import unicodedata


def resolve_nfc_filename(base_dir, expected_name):
    expected = unicodedata.normalize('NFC', expected_name)
    matches = [
        path for path in base_dir.iterdir()
        if path.is_file() and unicodedata.normalize('NFC', path.name) == expected
    ]
    assert len(matches) == 1, f'{expected_name}: 파일이 정확히 1개여야 합니다. 발견={matches}'
    return matches[0]


def read_csv_with_encodings(path, encodings=('utf-8-sig', 'utf-8', 'cp949')):
    errors = {}
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeDecodeError as exc:
            errors[encoding] = str(exc)
    raise UnicodeDecodeError('csv', b'', 0, 1, f'지원 인코딩 실패: {errors}')


FTP_PATH = resolve_nfc_filename(BASE_DIR, 'iM뱅크_월별_추정FTP_2023_2025.csv')
RATE_PATH = resolve_nfc_filename(BASE_DIR, '예대금리차2023~2025_순.csv')

FISIM_RATE_PATH = OUTPUT_DIR / 'fisim_monthly_rates_2023_2025.csv'
FISIM_MONTHLY_PATH = OUTPUT_DIR / 'fisim_customer_monthly_2023_2025.csv'
FISIM_INPUT_AUDIT_PATH = OUTPUT_DIR / 'fisim_direct_input_audit.csv'

# FTP: 제공 파일은 월 기준 소수(예: 0.002192)이다.
ftp_raw, ftp_encoding = read_csv_with_encodings(FTP_PATH)
ftp_raw.columns = [str(c).strip() for c in ftp_raw.columns]
assert {'기준년월', '월별 원가성자금 이자율'} <= set(ftp_raw.columns)
ftp = ftp_raw[['기준년월', '월별 원가성자금 이자율']].copy()
ftp['기준년월'] = pd.to_numeric(ftp['기준년월'], errors='raise').astype(int)
ftp['월'] = pd.PeriodIndex(ftp['기준년월'].astype(str), freq='M')
ftp['FTP_월_decimal'] = pd.to_numeric(ftp['월별 원가성자금 이자율'], errors='raise')
assert len(ftp) == 36 and ftp['월'].is_unique
assert ftp['FTP_월_decimal'].between(0, 0.02).all()

# 예대금리: 일부 내보내기 파일은 실제 헤더가 첫 데이터 행에 있으므로 승격한다.
rate_wide, rate_encoding = read_csv_with_encodings(RATE_PATH)
if {'은행', '구분'} - set(rate_wide.columns):
    promoted_header = rate_wide.iloc[0].astype(str).str.strip().tolist()
    rate_wide = rate_wide.iloc[1:].copy()
    rate_wide.columns = promoted_header
rate_wide.columns = [str(c).strip() for c in rate_wide.columns]
assert {'은행', '구분'} <= set(rate_wide.columns)
rate_wide['은행'] = rate_wide['은행'].ffill()

month_cols = [c for c in rate_wide.columns if re.fullmatch(r'\d{4}년\d{2}월', c)]
assert len(month_cols) == 36, f'금리 월 열 수 불일치: {len(month_cols)}'
selected_rates = rate_wide.loc[
    rate_wide['은행'].astype(str).str.contains('iM뱅크', na=False)
    & rate_wide['구분'].isin(['기업대출금리', '저축성수신금리']),
    ['은행', '구분'] + month_cols,
].copy()
assert selected_rates['구분'].value_counts().to_dict() == {
    '기업대출금리': 1, '저축성수신금리': 1
}

rate_long = selected_rates.melt(
    id_vars=['은행', '구분'], value_vars=month_cols,
    var_name='금리기준월', value_name='월율_pct',
)
date_parts = rate_long['금리기준월'].str.extract(r'(?P<year>\d{4})년(?P<month>\d{2})월')
rate_long['월'] = pd.PeriodIndex(date_parts['year'] + '-' + date_parts['month'], freq='M')
rate_long['월율_pct'] = pd.to_numeric(rate_long['월율_pct'], errors='raise')
rate_pivot = (
    rate_long.pivot(index='월', columns='구분', values='월율_pct')
    .reset_index().rename_axis(columns=None)
    .rename(columns={
        '기업대출금리': '기업대출금리_월_pct',
        '저축성수신금리': '저축성수신금리_월_pct',
    })
)

DEMAND_DEPOSIT_RATE_MONTHLY_PCT = 0.01
FISIM_RATES = ftp[['월', 'FTP_월_decimal']].merge(
    rate_pivot, on='월', how='inner', validate='one_to_one'
)
FISIM_RATES['기업대출금리_월_decimal'] = FISIM_RATES['기업대출금리_월_pct'] / 100
FISIM_RATES['저축성수신금리_월_decimal'] = FISIM_RATES['저축성수신금리_월_pct'] / 100
FISIM_RATES['요구불금리_월_pct'] = DEMAND_DEPOSIT_RATE_MONTHLY_PCT
FISIM_RATES['요구불금리_월_decimal'] = DEMAND_DEPOSIT_RATE_MONTHLY_PCT / 100
FISIM_RATES['대출스프레드_월'] = (
    FISIM_RATES['기업대출금리_월_decimal'] - FISIM_RATES['FTP_월_decimal']
)
FISIM_RATES['저축성수신스프레드_월'] = (
    FISIM_RATES['FTP_월_decimal'] - FISIM_RATES['저축성수신금리_월_decimal']
)
FISIM_RATES['요구불스프레드_월'] = (
    FISIM_RATES['FTP_월_decimal'] - FISIM_RATES['요구불금리_월_decimal']
)
FISIM_RATES = FISIM_RATES.sort_values('월').reset_index(drop=True)

assert len(FISIM_RATES) == 36
assert set(FISIM_RATES['월']) == set(PANEL['월'].unique())
assert FISIM_RATES.notna().all().all()
assert FISIM_RATES['기업대출금리_월_decimal'].between(0.002, 0.01).all()
assert FISIM_RATES['FTP_월_decimal'].between(0.001, 0.01).all()
np.testing.assert_allclose(
    FISIM_RATES['요구불금리_월_decimal'], 0.0001, rtol=0, atol=1e-15
)

rate_export = FISIM_RATES.copy()
rate_export.insert(0, '기준년월', rate_export['월'].astype(str).str.replace('-', '').astype(int))
rate_export['월'] = rate_export['월'].astype(str)
rate_export.to_csv(FISIM_RATE_PATH, index=False, encoding='utf-8-sig', float_format='%.10f')

RATE_SOURCE_AUDIT = pd.Series({
    '예대금리_파일': str(RATE_PATH),
    '예대금리_인코딩': rate_encoding,
    'FTP_파일': str(FTP_PATH),
    'FTP_인코딩': ftp_encoding,
    '월수': len(FISIM_RATES),
    '기업대출금리_단위': '월 기준 % -> /100',
    '저축성수신금리_단위': '월 기준 % -> /100',
    'FTP_단위': '월 기준 소수 -> 그대로',
    '요구불금리_가정': '월 0.01% = 0.0001',
})
display(RATE_SOURCE_AUDIT.to_frame('값'))
display(rate_export.head())

,값
예대금리_파일,/Users/changeun_1/Desktop/iM/5. final/예대금ᄅ...
예대금리_인코딩,cp949
FTP_파일,/Users/changeun_1/Desktop/iM/5. final/iM뱅크_...
FTP_인코딩,utf-8-sig
월수,36
기업대출금리_단위,월 기준 % -> /100
저축성수신금리_단위,월 기준 % -> /100
FTP_단위,월 기준 소수 -> 그대로
요구불금리_가정,월 0.01% = 0.0001


,기준년월,월,FTP_월_decimal,기업대출금리_월_pct,저축성수신금리_월_pct,기업대출금리_월_decimal,저축성수신금리_월_decimal,요구불금리_월_pct,요구불금리_월_decimal,대출스프레드_월,저축성수신스프레드_월,요구불스프레드_월
0,202301,2023-01,0.002192,0.479167,0.332500,0.004792,0.003325,0.01,0.0001,0.002600,-0.001133,0.002092
1,202302,2023-02,0.002192,0.474167,0.308333,0.004742,0.003083,0.01,0.0001,0.002550,-0.000892,0.002092
2,202303,2023-03,0.002192,0.456667,0.288333,0.004567,0.002883,0.01,0.0001,0.002375,-0.000692,0.002092
3,202304,2023-04,0.002217,0.450000,0.276667,0.004500,0.002767,0.01,0.0001,0.002283,-0.000550,0.002117
4,202305,2023-05,0.002217,0.459167,0.279167,0.004592,0.002792,0.01,0.0001,0.002375,-0.000575,0.002117


In [17]:
FISIM_BALANCE_COLS = [
    '여신_운전자금대출잔액', '여신_시설자금대출잔액',
    '거치식예금잔액', '적립식예금잔액', '요구불예금잔액',
]
missing_balance_cols = sorted(set(FISIM_BALANCE_COLS) - set(PANEL.columns))
assert not missing_balance_cols, f'필수 잔액 컬럼 누락: {missing_balance_cols}'

FISIM_MONTHLY = PANEL[['법인ID', '기준년월', '월'] + FISIM_BALANCE_COLS].copy()
for col in FISIM_BALANCE_COLS:
    FISIM_MONTHLY[col] = pd.to_numeric(FISIM_MONTHLY[col], errors='raise')
assert FISIM_MONTHLY[FISIM_BALANCE_COLS].notna().all().all()
assert FISIM_MONTHLY[FISIM_BALANCE_COLS].ge(0).all().all()
assert not FISIM_MONTHLY.duplicated(['법인ID', '월']).any()

FISIM_MONTHLY['대출잔액_L'] = (
    FISIM_MONTHLY['여신_운전자금대출잔액']
    + FISIM_MONTHLY['여신_시설자금대출잔액']
)
FISIM_MONTHLY['저축성수신잔액_DS'] = (
    FISIM_MONTHLY['거치식예금잔액']
    + FISIM_MONTHLY['적립식예금잔액']
)
FISIM_MONTHLY['요구불잔액_DR'] = FISIM_MONTHLY['요구불예금잔액']

FISIM_MONTHLY = FISIM_MONTHLY.merge(
    FISIM_RATES, on='월', how='left', validate='many_to_one'
)
required_rate_cols = [
    'FTP_월_decimal', '기업대출금리_월_decimal',
    '저축성수신금리_월_decimal', '요구불금리_월_decimal',
]
assert FISIM_MONTHLY[required_rate_cols].notna().all().all()

FISIM_MONTHLY['대출_FISIM_CONTRIB_M'] = (
    FISIM_MONTHLY['대출잔액_L'] * FISIM_MONTHLY['대출스프레드_월']
)
FISIM_MONTHLY['저축성수신_FISIM_CONTRIB_M'] = (
    FISIM_MONTHLY['저축성수신잔액_DS'] * FISIM_MONTHLY['저축성수신스프레드_월']
)
FISIM_MONTHLY['요구불_FISIM_CONTRIB_M'] = (
    FISIM_MONTHLY['요구불잔액_DR'] * FISIM_MONTHLY['요구불스프레드_월']
)
component_cols = [
    '대출_FISIM_CONTRIB_M', '저축성수신_FISIM_CONTRIB_M',
    '요구불_FISIM_CONTRIB_M',
]
FISIM_MONTHLY['FISIM_CONTRIB_M'] = FISIM_MONTHLY[component_cols].sum(axis=1)
np.testing.assert_allclose(
    FISIM_MONTHLY['FISIM_CONTRIB_M'],
    FISIM_MONTHLY[component_cols].sum(axis=1), rtol=0, atol=1e-10,
)

value_group = FISIM_MONTHLY.sort_values(['법인ID', '월']).groupby('법인ID', sort=False)['FISIM_CONTRIB_M']
FISIM_MONTHLY['FISIM_CONTRIB_6M'] = value_group.transform(
    lambda s: s.rolling(6, min_periods=6).sum()
)
FISIM_MONTHLY['FISIM_CONTRIB_12M'] = value_group.transform(
    lambda s: s.rolling(12, min_periods=12).sum()
)
FISIM_MONTHLY['FISIM_VALUE_SCOPE'] = 'FISIM_IMPLICIT_MARGIN_ONLY'
FISIM_MONTHLY['잔액기준'] = '월말잔액'
FISIM_MONTHLY['금리환산산식'] = '기업대출·저축성수신 월율_pct/100; FTP 월소수 그대로'

FISIM_MONTHLY_EXPORT = FISIM_MONTHLY.copy()
FISIM_MONTHLY_EXPORT['월'] = FISIM_MONTHLY_EXPORT['월'].astype(str)
FISIM_MONTHLY_EXPORT.to_csv(
    FISIM_MONTHLY_PATH, index=False, encoding='utf-8-sig', float_format='%.10f'
)

FISIM_DIRECT_INPUT_AUDIT = pd.DataFrame([
    ['대출잔액 L', '여신_운전자금대출잔액 + 여신_시설자금대출잔액', True, '대출 스프레드 가치'],
    ['저축성수신 DS', '거치식예금잔액 + 적립식예금잔액', True, '저축성수신 조달편익'],
    ['요구불 DR', '요구불예금잔액', True, '요구불 조달편익'],
    ['금리·FTP', '기업대출금리, 저축성수신금리, 요구불금리, FTP', True, '월별 금리마진'],
    ['R1 관계피처 16개', ', '.join(R1_FEATURES), False, '세그먼트 비교에만 사용'],
    ['D/A/C/K 동적피처 40개', ', '.join(DYNAMIC_FEATURES), False, '위험확률 모델에만 사용'],
], columns=['block', 'columns', 'direct_input_to_fisim_profitability', 'role'])
FISIM_DIRECT_INPUT_AUDIT.to_csv(
    FISIM_INPUT_AUDIT_PATH, index=False, encoding='utf-8-sig'
)

FISIM_AUDIT = pd.Series({
    '고객-월 행': len(FISIM_MONTHLY),
    '법인': FISIM_MONTHLY['법인ID'].nunique(),
    '월': FISIM_MONTHLY['월'].nunique(),
    '월기여가치_음수_행': int(FISIM_MONTHLY['FISIM_CONTRIB_M'].lt(0).sum()),
    '월기여가치_0_행': int(FISIM_MONTHLY['FISIM_CONTRIB_M'].eq(0).sum()),
    'FISIM_CONTRIB_M_합계': FISIM_MONTHLY['FISIM_CONTRIB_M'].sum(),
    '대출기여_합계': FISIM_MONTHLY['대출_FISIM_CONTRIB_M'].sum(),
    '저축성수신기여_합계': FISIM_MONTHLY['저축성수신_FISIM_CONTRIB_M'].sum(),
    '요구불기여_합계': FISIM_MONTHLY['요구불_FISIM_CONTRIB_M'].sum(),
})
display(FISIM_DIRECT_INPUT_AUDIT)
display(FISIM_AUDIT.to_frame('값'))
display(FISIM_MONTHLY[[
    '법인ID', '기준년월', '대출잔액_L', '저축성수신잔액_DS', '요구불잔액_DR',
    '대출_FISIM_CONTRIB_M', '저축성수신_FISIM_CONTRIB_M',
    '요구불_FISIM_CONTRIB_M', 'FISIM_CONTRIB_M',
]].tail(10))

,block,columns,direct_input_to_fisim_profitability,role
0,대출잔액 L,여신_운전자금대출잔액 + 여신_시설자금대출잔액,True,대출 스프레드 가치
1,저축성수신 DS,거치식예금잔액 + 적립식예금잔액,True,저축성수신 조달편익
2,요구불 DR,요구불예금잔액,True,요구불 조달편익
3,금리·FTP,"기업대출금리, 저축성수신금리, 요구불금리, FTP",True,월별 금리마진
4,R1 관계피처 16개,"핵심거래_수준, 수신자산_수준, 여신관계_수준, 수신상품_활성폭, 여신세부상품_활성...",False,세그먼트 비교에만 사용
5,D/A/C/K 동적피처 40개,"요구불_최근3대직전3_로그차이, 요구불_최근3대이전9_로그차이, 요구불_H2대H1_...",False,위험확률 모델에만 사용


,값
고객-월 행,121392.000000
법인,3372.000000
월,36.000000
월기여가치_음수_행,975.000000
월기여가치_0_행,263.000000
FISIM_CONTRIB_M_합계,432802.998696
대출기여_합계,398916.089531
저축성수신기여_합계,-7107.831541
요구불기여_합계,40994.740707


,법인ID,기준년월,대출잔액_L,저축성수신잔액_DS,요구불잔액_DR,대출_FISIM_CONTRIB_M,저축성수신_FISIM_CONTRIB_M,요구불_FISIM_CONTRIB_M,FISIM_CONTRIB_M
121382,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202503,1600.0,0.0,0.10,2.719999,-0.0,0.000199,2.720199
121383,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202504,1600.0,0.0,6.40,2.706666,-0.0,0.012267,2.718933
121384,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202505,1600.0,0.0,0.06,2.493333,-0.0,0.000115,2.493448
121385,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202506,1600.0,0.0,0.16,2.453333,-0.0,0.000307,2.453639
121386,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202507,1600.0,0.0,6.40,2.373333,-0.0,0.011787,2.385119
121387,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202508,1600.0,0.0,5.00,2.679999,-0.0,0.009208,2.689208
121388,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202509,1600.0,0.0,6.60,2.653333,-0.0,0.012155,2.665488
121389,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202510,1600.0,0.0,7.60,2.653333,-0.0,0.013617,2.666949
121390,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202511,1600.0,0.0,6.80,2.719999,-0.0,0.012183,2.732183
121391,ffed592946900dee9903846d0dd9b19d019eb81bdd2deb...,202512,1600.0,0.0,6.10,2.679999,-0.0,0.010929,2.690929


In [18]:
FISIM_MONTHLY['저축성수신_FISIM_CONTRIB_M'].value_counts()

저축성수신_FISIM_CONTRIB_M
-0.000000    107799
-0.055000        28
-0.011000        28
-0.048333        18
-0.069167        18
              ...  
-0.014950         1
-0.014025         1
-0.016425         1
-0.021846         1
-0.001533         1
Name: count, Length: 5963, dtype: int64

## 12. 미래 6개월 CLV와 잠재손실

예측 기준월 `c`에서 미래 `c+1~c+6`의 잔액은 **전년동월 잔액**을 기준선으로 사용하고, 금리는 기준월 `c`의 기업대출금리·저축성수신금리·FTP를 고정 적용한다.

\[
CLV^{NoRisk}_{i,c}=\sum_{h=1}^{6}\hat V^{FTP}_{i,c+h}
\]

\[
CLV^{Risk}_{i,c}=\sum_{h=1}^{6}\frac{\hat V^{FTP}_{i,c+h}S_{i,c,h}}{1+p_{i,c}},\qquad
PotentialLoss_{i,c}=CLV^{NoRisk}_{i,c}-CLV^{Risk}_{i,c}
\]

`risk_probability`는 향후 6개월 누적 지속거래약화 확률이므로 월별 일정 위험률을 가정해
`S(h)=(1-p)^(h/6)`으로 변환한다. 원천 CLV와 잠재손실은 음수를 보존하며, 방어순위는 `max(PotentialLoss, 0)`가 양수인 법인만 대상으로 한다.

검증과 운영은 분리한다.

- `2025.06` OOT: 순위를 먼저 산출하고 `2025.07~2025.12` 실제 FISIM을 사후 결합
- `2025.12` 운영: `2026.01~2026.06` 예측 기준선만 산출하며 성능평가로 해석하지 않음

In [19]:
def recent_profitability_summary(cutoff):
    cutoff = pd.Period(cutoff, freq='M')
    months = pd.period_range(cutoff - 11, cutoff, freq='M')
    history = FISIM_MONTHLY.loc[FISIM_MONTHLY['월'].isin(months)].copy()
    assert history['월'].nunique() == 12
    counts = history.groupby('법인ID')['월'].nunique()
    assert counts.eq(12).all()
    return (
        history.groupby('법인ID', as_index=False)
        .agg(
            최근12M_월수익성_평균=('FISIM_CONTRIB_M', 'mean'),
            최근12M_월수익성_중앙=('FISIM_CONTRIB_M', 'median'),
            최근12M_수익성_합계=('FISIM_CONTRIB_M', 'sum'),
            최근12M_음수월수=('FISIM_CONTRIB_M', lambda s: int(s.lt(0).sum())),
        )
    )


def rank_defense_priority(frame):
    out = frame.copy()
    out['잠재손실_방어대상'] = out['PotentialLoss'].clip(lower=0)
    out['방어대상여부'] = out['잠재손실_방어대상'].gt(0)
    out['defense_rank'] = pd.Series(pd.NA, index=out.index, dtype='Int64')
    eligible = out.loc[out['방어대상여부']].sort_values(
        ['잠재손실_방어대상', 'risk_probability', 'CLV_NoRisk', '법인ID'],
        ascending=[False, False, False, True], kind='mergesort',
    )
    out.loc[eligible.index, 'defense_rank'] = np.arange(1, len(eligible) + 1)
    return out


def build_six_month_clv(cutoff, score_frame, include_actual=False):
    cutoff = pd.Period(cutoff, freq='M')
    cutoff_label = str(cutoff)
    rate_row = FISIM_RATES.loc[FISIM_RATES['월'].eq(cutoff)]
    assert len(rate_row) == 1
    rate_row = rate_row.iloc[0]

    score_cols = [
        '법인ID', 'risk_probability', 'risk_rank', 'top10_flag',
        'current_segment', 'baseline_segment_2023', 'segment_transition',
        '업종_대분류', '업종_중분류',
    ]
    if 'Y_INTERVENE_M12_v2' in score_frame.columns:
        score_cols.append('Y_INTERVENE_M12_v2')
    missing = sorted(set(score_cols) - set(score_frame.columns))
    assert not missing, f'{cutoff_label} 점수 컬럼 누락: {missing}'
    scores = score_frame[score_cols].copy()
    scores['risk_probability'] = pd.to_numeric(scores['risk_probability'], errors='raise').clip(0, 1)
    assert scores['법인ID'].is_unique

    forecast_parts = []
    for h in range(1, 7):
        forecast_month = cutoff + h
        balance_reference_month = forecast_month - 12
        balances = FISIM_MONTHLY.loc[
            FISIM_MONTHLY['월'].eq(balance_reference_month),
            ['법인ID', '대출잔액_L', '저축성수신잔액_DS', '요구불잔액_DR'],
        ].copy()
        assert balances['법인ID'].is_unique
        balances['기준월'] = cutoff_label
        balances['예측개월차_h'] = h
        balances['예측대상월'] = str(forecast_month)
        balances['잔액참조월_전년동월'] = str(balance_reference_month)
        balances['대출스프레드_기준월'] = rate_row['대출스프레드_월']
        balances['저축성수신스프레드_기준월'] = rate_row['저축성수신스프레드_월']
        balances['요구불스프레드_기준월'] = rate_row['요구불스프레드_월']
        balances['예측_대출_FISIM'] = balances['대출잔액_L'] * rate_row['대출스프레드_월']
        balances['예측_저축성수신_FISIM'] = balances['저축성수신잔액_DS'] * rate_row['저축성수신스프레드_월']
        balances['예측_요구불_FISIM'] = balances['요구불잔액_DR'] * rate_row['요구불스프레드_월']
        balances['예측_FISIM_M'] = balances[[
            '예측_대출_FISIM', '예측_저축성수신_FISIM', '예측_요구불_FISIM'
        ]].sum(axis=1)
        forecast_parts.append(balances)

    monthly = pd.concat(forecast_parts, ignore_index=True)
    monthly = monthly.merge(
        scores[['법인ID', 'risk_probability']], on='법인ID', how='inner', validate='many_to_one'
    )
    monthly['S_사건미발생확률'] = np.power(
        1 - monthly['risk_probability'], monthly['예측개월차_h'] / 6.0
    )
    monthly['CLV_NoRisk_월기여'] = monthly['예측_FISIM_M']
    monthly['CLV_Risk_월기여'] = (
        monthly['예측_FISIM_M'] * monthly['S_사건미발생확률']
        / (1 + monthly['risk_probability'])
    )

    customer = (
        monthly.groupby('법인ID', as_index=False)
        .agg(
            CLV_NoRisk=('CLV_NoRisk_월기여', 'sum'),
            CLV_Risk=('CLV_Risk_월기여', 'sum'),
            예측월수=('예측개월차_h', 'size'),
            예측_FISIM_음수월수=('예측_FISIM_M', lambda s: int(s.lt(0).sum())),
        )
        .merge(scores, on='법인ID', how='left', validate='one_to_one')
        .merge(recent_profitability_summary(cutoff), on='법인ID', how='left', validate='one_to_one')
    )
    customer['기준월'] = cutoff_label
    customer['PotentialLoss'] = customer['CLV_NoRisk'] - customer['CLV_Risk']
    customer['CLV_VALUE_SCOPE'] = 'FISIM_IMPLICIT_MARGIN_ONLY'
    customer['CLV_HORIZON_MONTHS'] = 6
    customer['SURVIVAL_FORMULA'] = '(1-risk_probability)^(h/6)'
    customer['RISK_CLV_DENOMINATOR'] = '1+risk_probability'
    customer = rank_defense_priority(customer)

    if include_actual:
        actual_months = pd.period_range(cutoff + 1, cutoff + 6, freq='M')
        actual = FISIM_MONTHLY.loc[FISIM_MONTHLY['월'].isin(actual_months)].copy()
        assert actual['월'].nunique() == 6
        actual = (
            actual.groupby('법인ID', as_index=False)
            .agg(
                ACTUAL_FISIM_6M=('FISIM_CONTRIB_M', 'sum'),
                ACTUAL_FISIM_음수월수=('FISIM_CONTRIB_M', lambda s: int(s.lt(0).sum())),
            )
        )
        customer = customer.merge(actual, on='법인ID', how='left', validate='one_to_one')
        customer['OBSERVED_VALUE_CHANGE_VS_BASELINE'] = (
            customer['ACTUAL_FISIM_6M'] - customer['CLV_NoRisk']
        )
        customer['OBSERVED_VALUE_LOSS_VS_BASELINE'] = (
            customer['CLV_NoRisk'] - customer['ACTUAL_FISIM_6M']
        )
        customer['OBSERVED_POSITIVE_LOSS'] = customer['OBSERVED_VALUE_LOSS_VS_BASELINE'].clip(lower=0)

    assert customer['예측월수'].eq(6).all()
    assert customer[['CLV_NoRisk', 'CLV_Risk', 'PotentialLoss']].notna().all().all()
    zero_risk = customer['risk_probability'].eq(0)
    if zero_risk.any():
        np.testing.assert_allclose(
            customer.loc[zero_risk, 'PotentialLoss'], 0, rtol=0, atol=1e-8
        )
    positive_ranks = customer.loc[customer['방어대상여부'], 'defense_rank'].dropna().astype(int)
    assert positive_ranks.sort_values().tolist() == list(range(1, len(positive_ranks) + 1))
    return customer, monthly


TEST_CLV_202506, TEST_CLV_MONTHLY_202506 = build_six_month_clv(
    '2025-06', TEST_PREDICTIONS, include_actual=True
)
OPERATING_CLV_202512, OPERATING_CLV_MONTHLY_202512 = build_six_month_clv(
    '2025-12', OPERATING_SCORES, include_actual=False
)

display(TEST_CLV_202506.sort_values('defense_rank', na_position='last')[[
    '법인ID', 'current_segment', 'risk_probability', 'CLV_NoRisk', 'CLV_Risk',
    'PotentialLoss', '잠재손실_방어대상', 'defense_rank',
    'ACTUAL_FISIM_6M', 'OBSERVED_POSITIVE_LOSS',
]].head(20))
display(OPERATING_CLV_202512.sort_values('defense_rank', na_position='last')[[
    '법인ID', 'current_segment', 'risk_probability', 'CLV_NoRisk', 'CLV_Risk',
    'PotentialLoss', '잠재손실_방어대상', 'defense_rank',
]].head(20))

,법인ID,current_segment,risk_probability,CLV_NoRisk,CLV_Risk,PotentialLoss,잠재손실_방어대상,defense_rank,ACTUAL_FISIM_6M,OBSERVED_POSITIVE_LOSS
3144,f07470c0bc5e4923a36e92a6115a15b96aa65182cbdfd6...,복합고관계형,0.348178,559.024024,327.334154,231.689870,231.689870,1,225.966729,333.057295
3078,ea2815a6042eaf0d62d75ecc28a44477701bd93b34cac3...,복합고관계형,0.553846,197.564212,79.476778,118.087434,118.087434,2,207.764327,0.000000
3158,f17ef90681947a1973706993c549f27ac7d7e1a3d2b8a7...,복합고관계형,0.826087,108.789991,26.681073,82.108918,82.108918,3,26.472928,82.317063
952,46b84730d6c131995241bbc46ab4763f2d37b5fb468cdb...,복합고관계형,0.826087,101.897645,23.542300,78.355345,78.355345,4,99.018397,2.879248
437,209fc4344fe2fb4b3fbbf2a85b8bd5d39b6ff84352ce7a...,복합고관계형,0.170678,295.549943,226.671915,68.878029,68.878029,5,314.029944,0.000000
3060,e965d2014766e2d8a64d0bc2bb70cbe2ecd43fdf43174d...,거래·수신중심형,0.357513,144.916771,93.658037,51.258734,51.258734,6,-104.548062,249.464833
2266,ad3cc31636815a106d68af99561ee9288ae368f12c729f...,복합고관계형,0.134328,269.523912,219.028285,50.495627,50.495627,7,237.384987,32.138926
617,2d4951afe1a7311a8b1ebd1ad6dca146624a139b54123b...,복합고관계형,0.431818,91.999980,46.804884,45.195096,45.195096,8,98.499980,0.000000
431,2032b49e9b7a13f66433f73e88b12af219a803991dec04...,복합고관계형,0.714286,60.660572,18.585188,42.075384,42.075384,9,64.976322,0.000000
2843,d7f765af410ab5750e69acb3226b2bd1e2a106d3b105c6...,복합고관계형,0.046385,550.554715,511.802659,38.752055,38.752055,10,606.035087,0.000000


,법인ID,current_segment,risk_probability,CLV_NoRisk,CLV_Risk,PotentialLoss,잠재손실_방어대상,defense_rank
1048,4e7ad9901bab418169f38303bc4cac9e36e0f0d2dcac42...,복합고관계형,0.160828,1666.230658,1297.664996,368.565663,368.565663,1
1673,802f381c5cbe42155fa5118ba7d194ba8c07f056bcd4cc...,복합고관계형,0.277429,180.994564,118.724028,62.270536,62.270536,2
2251,ac40b5918adce9281816e04e19b55fb0cd5fda6e4cf806...,복합고관계형,0.205251,190.989740,139.126540,51.863200,51.863200,3
949,46b84730d6c131995241bbc46ab4763f2d37b5fb468cdb...,복합고관계형,0.362385,100.909752,57.437160,43.472593,43.472593,4
2730,cf48f05fa25c34b419766afcf6b566f9991483f05a2d6a...,저거래·저수신형,0.267760,105.533669,69.679922,35.853747,35.853747,5
1204,5a3ac356b3362a5427d17cfc88a0967b73c97294923364...,복합고관계형,0.073363,313.177368,280.247723,32.929645,32.929645,6
2027,9bc2277724a74a9abb792f39a5e437ec83afaf024b082d...,복합고관계형,0.631206,41.689855,14.939437,26.750418,26.750418,7
2493,be4c559f18493bab29c7f77bb1034e6dc884dd402509f8...,복합고관계형,0.857143,30.955369,6.161897,24.793473,24.793473,8
2838,d7f765af410ab5750e69acb3226b2bd1e2a106d3b105c6...,복합고관계형,0.024446,599.990049,577.294890,22.695159,22.695159,9
2848,d87669af0aee181d7dbb136ed8b283ebf812bb6f70fe4a...,복합고관계형,0.105550,134.009469,113.105483,20.903987,20.903987,10


#### 2025.06 OOT 해석 원칙

1. `OOT_MODEL_TOP5`는 당시 사용할 수 있었던 위험확률과 가치 기준의 방어대상이고, `OOT_ACTUAL_TOP5`는 미래를 본 뒤의 사후 상위손실이다.
2. 두 상위 5개의 중복률은 선택모형의 실질적 가치포착력을 보여주지만, 표본이 5개뿐이므로 장기 성능으로 일반화하지 않는다.
3. `기준선대비_실제차이 < 0`이면 실제 FISIM이 기준선보다 낮았다는 뜻이다.
4. 잔액효과가 크면 전년동월 잔액 재사용 가정의 오차가 주원인이고, 금리효과가 크면 2025.06 금리 고정 가정의 오차가 주원인이다.
5. `Y_INTERVENE_M12_v2=1`은 D와 A/C/K의 지속거래약화이며, FISIM 기준선 하회와 동일한 사건이 아니다. 실제손실 상위인데 Y=0일 수 있다.
6. 실제손실은 수수료·비용·신용손실을 포함하지 않은 FISIM 기준선 차이이며 확정 회계손실이 아니다.

## 13. 세그먼트 비교, OOT 가치검증, 운영 방어순위

세그먼트는 가치 산식이나 위험확률의 가중치가 아니다. 법인별 계산이 끝난 뒤 같은 기준월의 `current_segment`를 결합해 평균·중앙값·IQR·합계를 비교한다.

2025.06 OOT에서는 `risk_probability`와 방어순위를 2025.07~12 실제 가치보다 먼저 만든 것으로 간주하고, 실제 손실 노출의 사후 집중도를 확인한다. 2025.12는 아직 미래 실적이 없으므로 예측 CLV와 방어순위만 저장한다.

In [26]:
def segment_value_summary(frame, cutoff):
    summary = (
        frame.groupby('current_segment', as_index=False)
        .agg(
            법인=('법인ID', 'size'),
            위험확률_평균=('risk_probability', 'mean'),
            위험확률_중앙=('risk_probability', 'median'),
            최근12M_월수익성_평균=('최근12M_월수익성_평균', 'mean'),
            최근12M_월수익성_중앙=('최근12M_월수익성_중앙', 'median'),
            CLV_NoRisk_평균=('CLV_NoRisk', 'mean'),
            CLV_NoRisk_중앙=('CLV_NoRisk', 'median'),
            CLV_NoRisk_Q25=('CLV_NoRisk', lambda s: s.quantile(0.25)),
            CLV_NoRisk_Q75=('CLV_NoRisk', lambda s: s.quantile(0.75)),
            CLV_Risk_평균=('CLV_Risk', 'mean'),
            CLV_Risk_중앙=('CLV_Risk', 'median'),
            CLV_Risk_Q25=('CLV_Risk', lambda s: s.quantile(0.25)),
            CLV_Risk_Q75=('CLV_Risk', lambda s: s.quantile(0.75)),
            잠재손실_평균=('PotentialLoss', 'mean'),
            잠재손실_중앙=('PotentialLoss', 'median'),
            잠재손실_합계=('잠재손실_방어대상', 'sum'),
            방어대상_법인=('방어대상여부', 'sum'),
        )
    )
    summary.insert(0, '기준월', cutoff)
    return summary


SEGMENT_VALUE_202506 = segment_value_summary(TEST_CLV_202506, '2025-06')
SEGMENT_VALUE_202512 = segment_value_summary(OPERATING_CLV_202512, '2025-12')

def assign_decile_high_is_one(series):
    rank = series.rank(method='first', ascending=False)
    return pd.qcut(rank, 10, labels=False, duplicates='drop') + 1


TEST_CLV_202506['defense_decile'] = assign_decile_high_is_one(
    TEST_CLV_202506['잠재손실_방어대상']
)
TEST_VALUE_DECILES = (
    TEST_CLV_202506.groupby('defense_decile', as_index=False)
    .agg(
        법인=('법인ID', 'size'),
        양성=('Y_INTERVENE_M12_v2', 'sum'),
        위험확률_평균=('risk_probability', 'mean'),
        예측잠재손실_합계=('잠재손실_방어대상', 'sum'),
        실제양의손실_합계=('OBSERVED_POSITIVE_LOSS', 'sum'),
        실제FISIM_합계=('ACTUAL_FISIM_6M', 'sum'),
    )
)
actual_loss_total = TEST_VALUE_DECILES['실제양의손실_합계'].sum()
TEST_VALUE_DECILES['실제양의손실_포착률'] = np.where(
    actual_loss_total > 0,
    TEST_VALUE_DECILES['실제양의손실_합계'] / actual_loss_total,
    np.nan,
)
TEST_VALUE_DECILES['실제양의손실_누적포착률'] = TEST_VALUE_DECILES['실제양의손실_포착률'].cumsum()

top10_cut = max(1, math.ceil(TEST_CLV_202506['방어대상여부'].sum() * 0.10))
test_defense_sorted = TEST_CLV_202506.loc[TEST_CLV_202506['방어대상여부']].sort_values('defense_rank')
top10_actual_loss = test_defense_sorted.head(top10_cut)['OBSERVED_POSITIVE_LOSS'].sum()
TEST_VALUE_VALIDATION = pd.Series({
    '검증기준월': '2025-06',
    '사후확인기간': '2025-07~2025-12',
    '검증법인': len(TEST_CLV_202506),
    '방어대상법인': int(TEST_CLV_202506['방어대상여부'].sum()),
    '실제양의손실_합계': actual_loss_total,
    '상위10pct_실제양의손실_포착률': top10_actual_loss / actual_loss_total if actual_loss_total > 0 else np.nan,
    '위험확률_실제손실_Spearman': TEST_CLV_202506['risk_probability'].corr(
        TEST_CLV_202506['OBSERVED_POSITIVE_LOSS'], method='spearman'
    ),
    '예측잠재손실_실제손실_Spearman': TEST_CLV_202506['잠재손실_방어대상'].corr(
        TEST_CLV_202506['OBSERVED_POSITIVE_LOSS'], method='spearman'
    ),
    '운영결과와혼합금지': True,
})

output_frames = {
    OUTPUT_DIR / 'fisim_test_202506_clv_validation.csv': TEST_CLV_202506,
    OUTPUT_DIR / 'fisim_test_202506_forecast_monthly.csv': TEST_CLV_MONTHLY_202506,
    OUTPUT_DIR / 'fisim_test_202506_value_deciles.csv': TEST_VALUE_DECILES,
    OUTPUT_DIR / 'fisim_operating_202512_clv_defense_priority.csv': OPERATING_CLV_202512,
    OUTPUT_DIR / 'fisim_operating_202512_forecast_monthly.csv': OPERATING_CLV_MONTHLY_202512,
    OUTPUT_DIR / 'fisim_segment_value_summary_202506.csv': SEGMENT_VALUE_202506,
    OUTPUT_DIR / 'fisim_segment_value_summary_202512.csv': SEGMENT_VALUE_202512,
}
for path, frame in output_frames.items():
    frame.to_csv(path, index=False, encoding='utf-8-sig', float_format='%.10f')

display(SEGMENT_VALUE_202506)
display(SEGMENT_VALUE_202512)
display(TEST_VALUE_DECILES)
display(TEST_VALUE_VALIDATION.to_frame('값'))
print('2025.12 운영 방어순위:', OUTPUT_DIR / 'fisim_operating_202512_clv_defense_priority.csv')

,기준월,current_segment,법인,위험확률_평균,위험확률_중앙,최근12M_월수익성_평균,최근12M_월수익성_중앙,CLV_NoRisk_평균,CLV_NoRisk_중앙,CLV_NoRisk_Q25,CLV_NoRisk_Q75,CLV_Risk_평균,CLV_Risk_중앙,CLV_Risk_Q25,CLV_Risk_Q75,잠재손실_평균,잠재손실_중앙,잠재손실_합계,방어대상_법인
0,2025-06,거래·수신중심형,942,0.035521,0.007131,0.496916,0.310056,3.046559,1.931466,0.860871,3.309940,2.846266,1.813634,0.807808,3.089509,0.200293,0.018093,189.045206,881
1,2025-06,복합고관계형,1203,0.030254,0.004744,7.145026,3.084992,37.811728,16.766997,8.476889,32.920957,36.235331,15.953699,8.094356,31.806966,1.576397,0.157323,1896.405320,1138
2,2025-06,저거래·저수신형,1201,0.045646,0.007634,1.321919,0.502750,6.871027,2.621999,0.887915,6.890665,6.558353,2.433773,0.787319,6.433293,0.312674,0.045746,375.521771,1196


,기준월,current_segment,법인,위험확률_평균,위험확률_중앙,최근12M_월수익성_평균,최근12M_월수익성_중앙,CLV_NoRisk_평균,CLV_NoRisk_중앙,CLV_NoRisk_Q25,CLV_NoRisk_Q75,CLV_Risk_평균,CLV_Risk_중앙,CLV_Risk_Q25,CLV_Risk_Q75,잠재손실_평균,잠재손실_중앙,잠재손실_합계,방어대상_법인
0,2025-12,거래·수신중심형,934,0.025162,0.003397,0.614511,0.298135,3.144758,1.891075,0.807365,3.151645,3.046896,1.765652,0.784587,3.034512,0.097862,0.009204,95.990843,879
1,2025-12,복합고관계형,1180,0.022829,0.003397,6.472817,2.911100,38.990049,17.692178,9.182387,35.615403,37.705196,16.962486,8.821398,34.687895,1.284853,0.106869,1516.133915,1142
2,2025-12,저거래·저수신형,1227,0.053132,0.008260,1.218160,0.452499,7.448146,2.780499,0.901747,7.504290,7.084813,2.574039,0.808038,6.937521,0.363332,0.040697,445.808736,1225


,defense_decile,법인,양성,위험확률_평균,예측잠재손실_합계,실제양의손실_합계,실제FISIM_합계,실제양의손실_포착률,실제양의손실_누적포착률
0,1,335,50,0.169615,2074.503300,2750.794505,22344.999365,0.621866,0.621866
1,2,335,24,0.074954,202.716566,318.051332,9733.411509,0.071901,0.693767
2,3,334,9,0.037423,84.235559,435.797333,7523.779098,0.098520,0.792287
3,4,335,8,0.029797,45.018016,210.256414,5518.887547,0.047532,0.839819
4,5,334,7,0.021782,25.211042,178.814267,3354.297909,0.040424,0.880243
5,6,335,6,0.014255,14.883077,176.197517,3133.824637,0.039833,0.920076
6,7,334,4,0.009692,8.421120,90.840144,1847.363079,0.020536,0.940612
7,8,335,6,0.007393,4.052194,53.776394,1140.355592,0.012157,0.952769
8,9,334,3,0.004654,1.675902,51.310123,593.525657,0.011600,0.964369
9,10,335,2,0.002824,0.255520,157.613204,2631.068834,0.035631,1.000000


,값
검증기준월,2025-06
사후확인기간,2025-07~2025-12
검증법인,3346
방어대상법인,3215
실제양의손실_합계,4423.451234
상위10pct_실제양의손실_포착률,0.620194
위험확률_실제손실_Spearman,0.088589
예측잠재손실_실제손실_Spearman,0.031545
운영결과와혼합금지,True


2025.12 운영 방어순위: /Users/changeun_1/Desktop/iM/5. final/수익성F_outputs/fisim_operating_202512_clv_defense_priority.csv


## 14. 최종 감사 체크와 해석 원칙

1. 수익성 산식의 직접 입력은 여수신 월말잔액과 월별 금리·FTP뿐이다.
2. `risk_probability`는 실제 폐업·이탈확률이 아니라 향후 6개월 지속거래약화 확률이다.
3. `PotentialLoss`는 확정 회계손실이 아니라, 약화 시 가치가 감소한다는 가정 아래의 잠재 CLV 위험노출액이다.
4. 음수 FISIM은 역마진 진단을 위해 보존하며, 방어순위에서만 양의 잠재손실을 사용한다.
5. 2025.06의 사후검증과 2025.12의 미실현 운영예측을 같은 성과표로 혼합하지 않는다.
6. 수수료수익·운영원가·신용비용·자본비용은 자료에 없으므로 현재 값은 전체 순이익이 아니라 FISIM 기반 경제적 기여가치다.

In [27]:
CLV_CHECKS = pd.DataFrame([
    ['rate_months_36', len(FISIM_RATES), 36, len(FISIM_RATES) == 36],
    ['customer_month_rows', len(FISIM_MONTHLY), len(PANEL), len(FISIM_MONTHLY) == len(PANEL)],
    ['test_clv_rows', len(TEST_CLV_202506), len(TEST_PREDICTIONS), len(TEST_CLV_202506) == len(TEST_PREDICTIONS)],
    ['operating_clv_rows', len(OPERATING_CLV_202512), len(OPERATING_SCORES), len(OPERATING_CLV_202512) == len(OPERATING_SCORES)],
    ['test_forecast_6_months', int(TEST_CLV_202506['예측월수'].min()), 6, TEST_CLV_202506['예측월수'].eq(6).all()],
    ['operating_forecast_6_months', int(OPERATING_CLV_202512['예측월수'].min()), 6, OPERATING_CLV_202512['예측월수'].eq(6).all()],
    ['segment_not_direct_fisim_input', int(FISIM_DIRECT_INPUT_AUDIT.loc[
        FISIM_DIRECT_INPUT_AUDIT['block'].str.contains('피처'),
        'direct_input_to_fisim_profitability'
    ].sum()), 0, not FISIM_DIRECT_INPUT_AUDIT.loc[
        FISIM_DIRECT_INPUT_AUDIT['block'].str.contains('피처'),
        'direct_input_to_fisim_profitability'
    ].any()],
], columns=['check', 'actual', 'expected', 'pass'])
assert CLV_CHECKS['pass'].all(), CLV_CHECKS.loc[~CLV_CHECKS['pass']]
CLV_CHECKS.to_csv(
    OUTPUT_DIR / 'fisim_clv_checks.csv', index=False, encoding='utf-8-sig'
)

RUN_SUMMARY['fisim_clv_revised'] = {
    'rate_unit': 'loan/saving monthly percent /100; FTP monthly decimal unchanged',
    'demand_deposit_rate_monthly_decimal': 0.0001,
    'clv_horizon_months': 6,
    'survival_formula': '(1-risk_probability)^(h/6)',
    'test_cutoff': '2025-06',
    'operating_cutoff': '2025-12',
    'test_clv_rows': len(TEST_CLV_202506),
    'operating_clv_rows': len(OPERATING_CLV_202512),
    'operating_defense_targets': int(OPERATING_CLV_202512['방어대상여부'].sum()),
    'segment_used_as_value_weight': False,
}
with open(OUTPUT_DIR / 'run_summary.json', 'w', encoding='utf-8') as file:
    json.dump(RUN_SUMMARY, file, ensure_ascii=False, indent=2, default=str)

display(CLV_CHECKS)
display(pd.Series(RUN_SUMMARY['fisim_clv_revised']).to_frame('최종 잠금'))
print('전체 파이프라인과 수정본 수익성·CLV 계산 완료:', OUTPUT_DIR)

,check,actual,expected,pass
0,rate_months_36,36,36,True
1,customer_month_rows,121392,121392,True
2,test_clv_rows,3346,3346,True
3,operating_clv_rows,3341,3341,True
4,test_forecast_6_months,6,6,True
5,operating_forecast_6_months,6,6,True
6,segment_not_direct_fisim_input,0,0,True


,최종 잠금
rate_unit,loan/saving monthly percent /100; FTP monthly ...
demand_deposit_rate_monthly_decimal,0.0001
clv_horizon_months,6
survival_formula,(1-risk_probability)^(h/6)
test_cutoff,2025-06
operating_cutoff,2025-12
test_clv_rows,3346
operating_clv_rows,3341
operating_defense_targets,3246
segment_used_as_value_weight,False


전체 파이프라인과 수정본 수익성·CLV 계산 완료: /Users/changeun_1/Desktop/iM/5. final/수익성F_outputs
